# **HDFC Bank Service Request Resolution Analytics**


## Your Engineering Handover

You have joined the HDFC Bank Service Operations team as a Junior Python Developer. The operations lead spends hours switching between service request records, customer complaints, branch information, and request status updates to understand which cases are delayed, unresolved, or require immediate attention. You have been asked to build a reproducible service request resolution analysis workflow in this notebook.

Your deliverable is not an app or dashboard. It is a trustworthy set of cleaned data tables, a request-level investigation report, a branch SLA performance summary, a prioritized service resolution queue, and a concise evidence-based handover to Service Operations.

How to work: Make your own copy in Google Drive (`File → Save a copy in Drive`), upload the three provided files, complete the ten milestones in order, and run the assessment cell at the end. Write your own implementation in the code cells. Keep every required named output available for the next task and for assessment.

Input files (only three): `service_requests.csv` · `request_status_updates.xlsx` · `service_evidence.json` Data: Fictional training records; not actual HDFC Bank customer records.

> Important: The assessment at the end checks key output contracts and consistency. Passing it is not proof that every business calculation or written explanation is correct; reviewers may also inspect selected notebooks. Do not upload real banking or customer information to public AI tools.

### **Important Note: Guidelines for Using AI Tools**



You are allowed to use AI tools such as ChatGPT to understand concepts, debug errors, and get assistance with **specific code snippets or individual problems**.

However, **using AI to generate the complete project solution in one go is not permitted**.

You are expected to develop the solution milestone by milestone, apply your understanding of Python and Pandas, and independently verify your code and outputs.

**Remember:** The goal of this project is not just to complete the tasks but to develop practical problem-solving, debugging, and data analysis skills that are expected in real-world development roles.

## **Workspace setup · Data intake**
Run the setup cell and upload the **three input files** when requested. Files stay in your current Colab session; if the runtime restarts, upload them again. Keep original filenames unchanged.

[Link to Dataset](https://drive.google.com/file/d/1ikginmnDgXAmWQvSDnfoNMmZWSqsHHzV/view?usp=sharing)

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import zipfile
DATA_DIR = Path('/content') if Path('/content').is_dir() else Path.cwd()
REQUIRED_FILES = ['service_requests.csv', 'service_evidence.json', 'request_status_updates.xlsx']
DATASET_ZIP = 'hdfc_service_request_project_datasets.zip'
missing = [name for name in REQUIRED_FILES if not (DATA_DIR/name).is_file()]
if missing and (DATA_DIR/DATASET_ZIP).is_file():
    with zipfile.ZipFile(DATA_DIR/DATASET_ZIP) as archive:
        archive.extractall(DATA_DIR)
    missing = [name for name in REQUIRED_FILES if not (DATA_DIR/name).is_file()]
if missing and DATA_DIR.as_posix() == '/content':
    from google.colab import files
    print('Upload the dataset ZIP or all three files (CSV, JSON and XLSX):')
    files.upload()
    if (DATA_DIR/DATASET_ZIP).is_file():
        with zipfile.ZipFile(DATA_DIR/DATASET_ZIP) as archive:
            archive.extractall(DATA_DIR)
    missing = [name for name in REQUIRED_FILES if not (DATA_DIR/name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing required HDFC files in {DATA_DIR}: {missing}')
print('HDFC files ready:', len(REQUIRED_FILES), '| Directory:', DATA_DIR)


Upload the dataset ZIP or all three files (CSV, JSON and XLSX):


Saving hdfc_service_request_project_datasets.zip to hdfc_service_request_project_datasets.zip
HDFC files ready: 3 | Directory: /content


## Milestone 01 · Establish the service request baseline

**From the team:** The service operations lead needs a trusted intake snapshot before any service-level calculations.

**Your responsibility:** Load the request CSV and inspect raw rows, unique request IDs, customers, categories, missing values, and data types.

**Handover evidence:** `requests_df` as the raw request DataFrame; show row count, distinct IDs, schema, missingness and a preview.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import zipfile

DATA_DIR = Path('/content')


REQUIRED_FILES = [
    'service_requests.csv',
    'service_evidence.json',
    'request_status_updates.xlsx'
]

DATASET_ZIP = 'hdfc_service_request_project_datasets.zip'

missing = [
    name for name in REQUIRED_FILES
    if not (DATA_DIR / name).is_file()
]

if missing and (DATA_DIR / DATASET_ZIP).is_file():
    with zipfile.ZipFile(DATA_DIR / DATASET_ZIP) as archive:
        archive.extractall(DATA_DIR)

missing = [
    name for name in REQUIRED_FILES
    if not (DATA_DIR / name).is_file()
]

if missing:
    raise FileNotFoundError(
        f'Missing required files: {missing}'
    )

print(
    'HDFC files ready:',
    len(REQUIRED_FILES),
    '| Directory:',
    DATA_DIR
)

requests_df = pd.read_csv(
    DATA_DIR / 'service_requests.csv'
)

print("Dataset loaded successfully!")


HDFC files ready: 3 | Directory: /content
Dataset loaded successfully!


## Milestone 02 · Onboard branch and service evidence

**From the team:** The branch desk, ticket desk, and escalation team each use a different export format.

**Your responsibility:** Load customers, branches, support tickets, status-update spreadsheet and complaint JSON. Inspect counts, statuses and relationships. Read contextual TXT notes.

**Handover evidence:** `customers_df`, `branches_df`, `tickets_df`, `updates_df`, `complaints_df` and a short input QA summary.

**Input packaging:** Three files only: the JSON holds four logically separate entity/event collections, and the XLSX holds status updates.

In [4]:
from pathlib import Path
import pandas as pd
import json
import zipfile


DATA_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()

REQUIRED_FILES = [
    "service_requests.csv",
    "service_evidence.json",
    "request_status_updates.xlsx"
]

DATASET_ZIP = DATA_DIR / "hdfc_service_request_project_datasets.zip"

missing_files = [
    filename for filename in REQUIRED_FILES
    if not (DATA_DIR / filename).is_file()
]

if missing_files and DATASET_ZIP.is_file():
    with zipfile.ZipFile(DATASET_ZIP, "r") as archive:
        archive.extractall(DATA_DIR)

missing_files = [
    filename for filename in REQUIRED_FILES
    if not (DATA_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing required files in {DATA_DIR}: {missing_files}"
    )

print("All three required input files are available.")


json_path = DATA_DIR / "service_evidence.json"

with open(json_path, "r", encoding="utf-8") as file:
    evidence = json.load(file)

expected_collections = {
    "customers",
    "branches",
    "support_tickets",
    "customer_complaints"
}

if not isinstance(evidence, dict):
    raise ValueError("The JSON file must contain a dictionary.")

if set(evidence.keys()) != expected_collections:
    raise ValueError(
        f"Expected exactly four collections: {expected_collections}. "
        f"Found: {set(evidence.keys())}"
    )

print("\nJSON validation successful.")
print("Collections:", list(evidence.keys()))


customers_df = pd.DataFrame(evidence["customers"])

branches_df = pd.DataFrame(evidence["branches"])

tickets_df = pd.DataFrame(evidence["support_tickets"])

complaints_df = pd.DataFrame(evidence["customer_complaints"])


excel_path = DATA_DIR / "request_status_updates.xlsx"

updates_df = pd.read_excel(excel_path)

print("\nExcel status updates loaded successfully.")


dataframes = {
    "customers_df": customers_df,
    "branches_df": branches_df,
    "tickets_df": tickets_df,
    "updates_df": updates_df,
    "complaints_df": complaints_df
}

for name, df in dataframes.items():

    print(name)


    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Column names:", df.columns.tolist())

    display(df.head(3))


qa_records = []

for name, df in dataframes.items():
    qa_records.append({
        "dataset": name,
        "row_count": len(df),
        "column_count": len(df.columns),
        "total_missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

qa_summary_df = pd.DataFrame(qa_records)

print("\nINPUT QUALITY SUMMARY")
display(qa_summary_df)


missing_records = []

for name, df in dataframes.items():
    for column in df.columns:
        missing_records.append({
            "dataset": name,
            "column": column,
            "missing_count": int(df[column].isna().sum())
        })

missing_values_df = pd.DataFrame(missing_records)

print("\nCOLUMN-WISE MISSING VALUE SUMMARY")
display(
    missing_values_df[
        missing_values_df["missing_count"] > 0
    ].reset_index(drop=True)
)


print("\nSUPPORT TICKET STATUS DISTRIBUTION")

if "ticket_status" in tickets_df.columns:
    ticket_status_summary = (
        tickets_df["ticket_status"]
        .value_counts(dropna=False)
        .rename_axis("ticket_status")
        .reset_index(name="count")
    )

    display(ticket_status_summary)
else:
    print("ticket_status column is missing.")


print("\nCUSTOMER COMPLAINT STATUS DISTRIBUTION")

if "complaint_status" in complaints_df.columns:
    complaint_status_summary = (
        complaints_df["complaint_status"]
        .value_counts(dropna=False)
        .rename_axis("complaint_status")
        .reset_index(name="count")
    )

    display(complaint_status_summary)
else:
    print("complaint_status column is missing.")

identifier_columns = {
    "customers_df": (customers_df, "customer_id"),
    "branches_df": (branches_df, "branch_id"),
    "tickets_df": (tickets_df, "ticket_id"),
    "updates_df": (updates_df, "update_id"),
    "complaints_df": (complaints_df, "complaint_id")
}

identifier_records = []

for name, (df, column) in identifier_columns.items():

    if column not in df.columns:
        print(f"WARNING: {column} is missing from {name}.")
        continue

    identifier_records.append({
        "dataset": name,
        "identifier_column": column,
        "unique_identifiers": int(df[column].nunique()),
        "missing_identifiers": int(df[column].isna().sum()),
        "duplicate_identifiers": int(
            df[column].duplicated(keep="first").sum()
        )
    })

identifier_qa_df = pd.DataFrame(identifier_records)

print("\nIDENTIFIER QUALITY SUMMARY")
display(identifier_qa_df)

def normalize_id(series):
    """Convert identifiers to comparable strings."""
    return series.astype("string").str.strip()


relationship_records = []

if "requests_df" in globals():

    valid_request_ids = set(
        normalize_id(
            requests_df["request_id"].dropna()
        ).tolist()
    )

    valid_customer_ids = set(
        normalize_id(
            customers_df["customer_id"].dropna()
        ).tolist()
    )

    valid_branch_ids = set(
        normalize_id(
            branches_df["branch_id"].dropna()
        ).tolist()
    )

    activity_datasets = {
        "tickets_df": tickets_df,
        "updates_df": updates_df,
        "complaints_df": complaints_df
    }

    for name, df in activity_datasets.items():

        if "request_id" not in df.columns:
            print(f"WARNING: request_id missing from {name}.")
            continue

        request_ids = normalize_id(df["request_id"])

        missing_count = int(request_ids.isna().sum())

        unmatched_count = int(
            (
                request_ids.notna()
                & ~request_ids.isin(valid_request_ids)
            ).sum()
        )

        relationship_records.append({
            "dataset": name,
            "total_records": len(df),
            "missing_request_ids": missing_count,
            "unmatched_request_ids": unmatched_count
        })

    customer_datasets = {
        "requests_df": requests_df,
        "tickets_df": tickets_df,
        "complaints_df": complaints_df
    }

    print("\nCUSTOMER REFERENCE VALIDATION")

    for name, df in customer_datasets.items():

        if "customer_id" not in df.columns:
            continue

        customer_ids = normalize_id(df["customer_id"])

        unmatched_count = int(
            (
                customer_ids.notna()
                & ~customer_ids.isin(valid_customer_ids)
            ).sum()
        )

        print(
            f"{name}: {unmatched_count} records with "
            "unmatched customer IDs."
        )

    request_branches = normalize_id(requests_df["branch_id"])

    unmatched_branches = int(
        (
            request_branches.notna()
            & ~request_branches.isin(valid_branch_ids)
        ).sum()
    )

    print(
        "\nService requests with unmatched branch IDs:",
        unmatched_branches
    )

    if "preferred_branch_id" in customers_df.columns:

        preferred_branches = normalize_id(
            customers_df["preferred_branch_id"]
        )

        unmatched_preferred_branches = int(
            (
                preferred_branches.notna()
                & ~preferred_branches.isin(valid_branch_ids)
            ).sum()
        )

        print(
            "Customers with unmatched preferred branch IDs:",
            unmatched_preferred_branches
        )

else:
    print(
        "\nRelationship validation skipped: "
        "Run Milestone 01 first to create requests_df."
    )


relationship_qa_df = pd.DataFrame(relationship_records)

print("\nREQUEST RELATIONSHIP QUALITY SUMMARY")
display(relationship_qa_df)

notes_path = DATA_DIR / "status_notes.txt"

if notes_path.is_file():

    operational_notes = notes_path.read_text(encoding="utf-8")

    print("\nOPERATIONAL STATUS NOTES")
    print(operational_notes)

else:

    operational_notes = None

    print(
        "\nOperational notes unavailable: "
        "status_notes.txt was not included among the supplied files."
    )

print("MILESTONE 02 COMPLETED")


print("Required DataFrames created:")

for name in dataframes:
    print(f"  {name}: {len(dataframes[name])} rows")

print("\nAdditional QA outputs created:")
print("  qa_summary_df")
print("  missing_values_df")
print("  identifier_qa_df")
print("  relationship_qa_df")

print("\nAll DataFrames are available for downstream milestones.")

All three required input files are available.

JSON validation successful.
Collections: ['customers', 'branches', 'support_tickets', 'customer_complaints']

Excel status updates loaded successfully.
customers_df
Rows: 1200
Columns: 6
Column names: ['customer_id', 'customer_name', 'customer_segment', 'account_type', 'relationship_age_days', 'preferred_branch_id']


,customer_id,customer_name,customer_segment,account_type,relationship_age_days,preferred_branch_id
0,CUST000001,Priya Jain,imperia,NRI Savings,2009,BR0605
1,CUST000002,Vihaan Jain,NRI,Loan Account,1056,BR0597
2,CUST000003,Ira Jain,Senior Citizen,Current,3331,BR0798


branches_df
Rows: 1000
Columns: 6
Column names: ['branch_id', 'branch_name', 'branch_city', 'branch_state', 'branch_zone', 'branch_type']


,branch_id,branch_name,branch_city,branch_state,branch_zone,branch_type
0,BR0001,HDFC Ahmedabad Service Branch 0001,Ahmedabad,Gujarat,West,Metro
1,BR0002,HDFC Kanpur Service Branch 0002,Kanpur,Uttar Pradesh,North,Semi-Urban
2,BR0003,HDFC Salem Service Branch 0003,Salem,Tamil Nadu,South,Metro


tickets_df
Rows: 1500
Columns: 8
Column names: ['ticket_id', 'request_id', 'customer_id', 'ticket_created_date', 'ticket_status', 'assigned_team', 'resolution_time_hours', 'escalation_flag']


,ticket_id,request_id,customer_id,ticket_created_date,ticket_status,assigned_team,resolution_time_hours,escalation_flag
0,TKT000001,REQ0001676,CUST000879,2026-02-10,Pending,Digital Banking,148.0,No
1,TKT000002,REQ0000292,CUST000404,2026-05-27,Closed,Digital Banking,56.0,yes
2,TKT000003,REQ0003061,CUST000604,2026-02-06,Resolved,Digital Banking,NaN,No


... (13 outputs left out)

,dataset,identifier_column,unique_identifiers,missing_identifiers,duplicate_identifiers
0,customers_df,customer_id,1200,0,0
1,branches_df,branch_id,1000,0,0
2,tickets_df,ticket_id,1500,0,0
3,updates_df,update_id,1300,0,0
4,complaints_df,complaint_id,1500,0,0



CUSTOMER REFERENCE VALIDATION
requests_df: 14 records with unmatched customer IDs.
tickets_df: 24 records with unmatched customer IDs.
complaints_df: 21 records with unmatched customer IDs.

Service requests with unmatched branch IDs: 11
Customers with unmatched preferred branch IDs: 0

REQUEST RELATIONSHIP QUALITY SUMMARY


,dataset,total_records,missing_request_ids,unmatched_request_ids
0,tickets_df,1500,0,31
1,updates_df,1300,0,30
2,complaints_df,1500,0,29



Operational notes unavailable: status_notes.txt was not included among the supplied files.
MILESTONE 02 COMPLETED
Required DataFrames created:
  customers_df: 1200 rows
  branches_df: 1000 rows
  tickets_df: 1500 rows
  updates_df: 1300 rows
  complaints_df: 1500 rows

Additional QA outputs created:
  qa_summary_df
  missing_values_df
  identifier_qa_df
  relationship_qa_df

All DataFrames are available for downstream milestones.


## Milestone 03 · Prepare dependable service requests

**From the team:** Duplicate IDs and inconsistent status labels can inflate SLA breaches and obscure unresolved requests.

**Your responsibility:** Keep one record per request ID, normalize category/channel/status, parse creation and resolution dates/times, convert SLA and monetary values to numeric, and identify invalid closure dates.

**Handover evidence:** `requests_clean`, one row per `request_id`, with `created_at`, `resolved_at`, `is_resolved`, `request_date_issue`.

In [5]:
import pandas as pd
import numpy as np

requests_work = requests_df.copy()

initial_rows = len(requests_work)
initial_unique_ids = requests_work["request_id"].nunique()

requests_work["request_id"] = (
    requests_work["request_id"]
    .astype("string")
    .str.strip()
)

requests_work["request_id"] = requests_work["request_id"].replace(
    "", pd.NA
)

duplicate_requests = requests_work[
    requests_work["request_id"].notna()
    & requests_work["request_id"].duplicated(keep=False)
].sort_values("request_id")

print("Initial row count:", initial_rows)
print("Initial unique request IDs:", initial_unique_ids)
print("Rows belonging to duplicate-ID groups:", len(duplicate_requests))

if not duplicate_requests.empty:
    display(duplicate_requests)


requests_clean = requests_work.dropna(
    subset=["request_id"]
).drop_duplicates(
    subset=["request_id"],
    keep="first"
).copy()

print("\nRows after duplicate removal:", len(requests_clean))
print("Unique request IDs:", requests_clean["request_id"].nunique())



text_columns = [
    "request_category",
    "channel",
    "request_status",
    "request_subcategory",
    "assigned_team"
]

for column in text_columns:
    if column in requests_clean.columns:
        requests_clean[column] = (
            requests_clean[column]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

for column in ["request_category", "channel"]:
    if column in requests_clean.columns:
        requests_clean[column] = (
            requests_clean[column].str.title()
        )

if "request_status" in requests_clean.columns:
    requests_clean["request_status"] = (
        requests_clean["request_status"].str.lower()
    )

    status_mapping = {
        "in progress": "in_progress",
        "in-progress": "in_progress",
        "work in progress": "in_progress",
        "pending": "pending",
        "open": "open",
        "resolved": "resolved",
        "closed": "closed",
        "reopened": "reopened",
        "cancelled": "cancelled",
        "canceled": "cancelled"
    }

    requests_clean["request_status"] = (
        requests_clean["request_status"]
        .replace(status_mapping)
    )



numeric_columns = ["sla_hours", "reported_amount"]

for column in numeric_columns:
    if column in requests_clean.columns:
        requests_clean[column] = pd.to_numeric(
            requests_clean[column]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.strip(),
            errors="coerce"
        )

requests_clean["sla_invalid"] = (
    requests_clean["sla_hours"].isna()
    | (requests_clean["sla_hours"] <= 0)
)

print("\nSLA values requiring investigation:",
      int(requests_clean["sla_invalid"].sum()))


created_date = (
    requests_clean["request_created_date"]
    .astype("string").str.strip()
)

created_time = (
    requests_clean["request_created_time"]
    .astype("string").str.strip()
)

created_text = created_date + " " + created_time

requests_clean["created_at"] = pd.to_datetime(
    created_text,
    errors="coerce"
)

requests_clean["resolved_at"] = pd.to_datetime(
    requests_clean["resolved_date"],
    errors="coerce"
)

requests_clean["last_updated_at"] = pd.to_datetime(
    requests_clean["last_updated_date"],
    errors="coerce"
)


resolved_statuses = {"resolved", "closed"}

requests_clean["is_resolved"] = (
    requests_clean["request_status"]
    .isin(resolved_statuses)
)

missing_created_date = requests_clean["created_at"].isna()

missing_resolution_date = (
    requests_clean["is_resolved"]
    & requests_clean["resolved_at"].isna()
)

resolution_before_creation = (
    requests_clean["resolved_at"].notna()
    & requests_clean["created_at"].notna()
    & (
        requests_clean["resolved_at"]
        < requests_clean["created_at"]
    )
)

unresolved_with_resolution_date = (
    ~requests_clean["is_resolved"]
    & requests_clean["resolved_at"].notna()
)

requests_clean["request_date_issue"] = (
    missing_created_date
    | missing_resolution_date
    | resolution_before_creation
    | unresolved_with_resolution_date
)



def identify_date_issues(row):
    issues = []

    if pd.isna(row["created_at"]):
        issues.append("missing_or_invalid_created_at")

    if row["is_resolved"] and pd.isna(row["resolved_at"]):
        issues.append("missing_or_invalid_resolved_at")

    if (
        pd.notna(row["created_at"])
        and pd.notna(row["resolved_at"])
        and row["resolved_at"] < row["created_at"]
    ):
        issues.append("resolution_before_creation")

    if not row["is_resolved"] and pd.notna(row["resolved_at"]):
        issues.append("unresolved_status_with_resolution_date")

    return "; ".join(issues) if issues else "none"


requests_clean["request_date_issue_reason"] = (
    requests_clean.apply(identify_date_issues, axis=1)
)



print("MILESTONE 03: DATA QUALITY SUMMARY")

print("Original rows:", initial_rows)
print("Rows retained:", len(requests_clean))
print("Unique request IDs:", requests_clean["request_id"].nunique())
print(
    "Duplicate IDs remaining:",
    int(requests_clean["request_id"].duplicated().sum())
)
print(
    "Missing request IDs excluded:",
    int(requests_work["request_id"].isna().sum())
)
print(
    "Resolved requests:",
    int(requests_clean["is_resolved"].sum())
)
print(
    "Unresolved requests:",
    int((~requests_clean["is_resolved"]).sum())
)
print(
    "Requests with date issues:",
    int(requests_clean["request_date_issue"].sum())
)

print("\nRequest status distribution:")
display(
    requests_clean["request_status"]
    .value_counts(dropna=False)
    .rename_axis("request_status")
    .reset_index(name="count")
)

print("\nDate quality summary:")
display(
    requests_clean["request_date_issue_reason"]
    .value_counts()
    .rename_axis("date_issue")
    .reset_index(name="count")
)

print("\nCleaned service request preview:")
display(
    requests_clean[
        [
            "request_id",
            "request_category",
            "channel",
            "request_status",
            "sla_hours",
            "created_at",
            "resolved_at",
            "last_updated_at",
            "is_resolved",
            "request_date_issue",
            "request_date_issue_reason"
        ]
    ].head(10)
)


assert requests_clean["request_id"].is_unique
assert "created_at" in requests_clean.columns
assert "resolved_at" in requests_clean.columns
assert "is_resolved" in requests_clean.columns
assert "request_date_issue" in requests_clean.columns

print("\nMilestone 03 handover checks passed.")

Initial row count: 5000
Initial unique request IDs: 4992
Rows belonging to duplicate-ID groups: 16


,request_id,customer_id,branch_id,request_created_date,request_created_time,request_category,request_subcategory,channel,request_status,sla_hours,assigned_team,reported_amount,last_updated_date,resolved_date
423,REQ0000432,CUST000049,BR0109,2026-06-05,04:21:00,digital banking,Beneficiary Issue,WhatsApp,Closed,24,Digital Banking Support,10010.29,2026-06-06,2026-06-06
4996,REQ0000432,CUST000049,BR0109,2026-06-05,04:21:00,digital banking,Beneficiary Issue,WhatsApp,in_progress,24,Digital Banking Support,10010.29,2026-06-06,2026-06-06
943,REQ0000952,CUST000073,BR0204,2026-06-20,10:27:00,Card,Chargeback,Branch,Pending Customer,36,Cards Operations,0.0,2026-06-22,NaN
4993,REQ0000952,CUST000073,BR0204,2026-06-20,10:27:00,Card,Chargeback,Branch,in_progress,36,Cards Operations,0.0,2026-06-22,NaN
1642,REQ0001651,CUST000742,BR0292,2026-05-03,08:02:00,Complaint,Fraud Concern,call center,In Progress,72,Customer Grievance,0.0,2026-05-06,NaN
4992,REQ0001651,CUST000742,BR0292,2026-05-03,08:02:00,Complaint,Fraud Concern,call center,in_progress,72,Customer Grievance,0.0,2026-05-06,NaN
1760,REQ0001769,CUST000433,BR0516,2026-03-09,06:10:00,Card,Chargeback,Call Center,Pending Customer,36,Cards Operations,26071.49,2026-03-11,NaN
4997,REQ0001769,CUST000433,BR0516,2026-03-09,06:10:00,Card,Chargeback,Call Center,in_progress,36,Cards Operations,26071.49,2026-03-11,NaN
2221,REQ0002230,CUST001040,BR0461,2026-03-18,10:07:00,Card,Failed ATM Withdrawal,Email,In Progress,36,Cards Operations,0.0,2026-03-20,NaN
4995,REQ0002230,CUST001040,BR0461,2026-03-18,10:07:00,Card,Failed ATM Withdrawal,Email,in_progress,36,Cards Operations,0.0,2026-03-20,NaN



Rows after duplicate removal: 4992
Unique request IDs: 4992

SLA values requiring investigation: 0
MILESTONE 03: DATA QUALITY SUMMARY
Original rows: 5000
Rows retained: 4992
Unique request IDs: 4992
Duplicate IDs remaining: 0
Missing request IDs excluded: 0
Resolved requests: 2360
Unresolved requests: 2632
Requests with date issues: 289

Request status distribution:


,request_status,count
0,resolved,1463
1,in_progress,1243
2,closed,897
3,open,550
4,pending customer,495
5,escalated,323
6,pending_customer,21



Date quality summary:


,date_issue,count
0,none,4703
1,resolution_before_creation,279
2,missing_or_invalid_created_at,6
3,resolution_before_creation; unresolved_status_...,2
4,missing_or_invalid_resolved_at,2



Cleaned service request preview:


,request_id,request_category,channel,request_status,sla_hours,created_at,resolved_at,last_updated_at,is_resolved,request_date_issue,request_date_issue_reason
0,REQ0000009,Complaint,Branch,in_progress,72,2026-04-27 04:20:00,NaT,2026-04-27,False,False,none
1,REQ0000010,Card,Whatsapp,pending customer,36,2026-05-17 22:31:00,NaT,2026-05-18,False,False,none
2,REQ0000011,Digital Banking,Email,closed,24,2026-06-19 04:47:00,2026-06-19,2026-06-19,True,True,resolution_before_creation
3,REQ0000012,Complaint,Email,in_progress,72,2026-04-20 03:36:00,NaT,2026-04-21,False,False,none
4,REQ0000013,Account,Netbanking,closed,48,2026-06-13 07:18:00,2026-06-16,2026-06-16,True,False,none
5,REQ0000014,Kyc,Call Center,closed,72,2026-04-25 06:03:00,2026-04-26,2026-04-26,True,False,none
6,REQ0000015,Account,Email,resolved,48,2026-02-22 19:15:00,2026-02-24,2026-02-25,True,False,none
7,REQ0000016,Account,Branch,resolved,48,2026-05-25 15:46:00,2026-05-27,2026-05-27,True,False,none
8,REQ0000017,Digital Banking,Whatsapp,pending customer,24,2026-04-21 18:52:00,NaT,2026-04-22,False,False,none
9,REQ0000018,Complaint,Whatsapp,resolved,72,2026-04-24 02:08:00,2026-04-28,2026-04-28,True,False,none



Milestone 03 handover checks passed.


## Milestone 04 · Standardize customer and event records

**From the team:** A request can legitimately have many updates, tickets and complaints; dropping repeated request IDs would erase evidence.

**Your responsibility:** Dedupe only by each table’s own unique ID, normalize event statuses and flags, parse dates, and check unmatched foreign keys.

**Handover evidence:** `customers_clean`, `branches_clean`, `tickets_clean`, `updates_clean`, `complaints_clean`, with retained distinct events.

In [6]:
import pandas as pd
import numpy as np

customers_clean = customers_df.copy()
branches_clean = branches_df.copy()
tickets_clean = tickets_df.copy()
updates_clean = updates_df.copy()
complaints_clean = complaints_df.copy()

original_counts = {
    "customers": len(customers_clean),
    "branches": len(branches_clean),
    "tickets": len(tickets_clean),
    "updates": len(updates_clean),
    "complaints": len(complaints_clean)
}

identifier_columns = {
    "customers_clean": ["customer_id", "preferred_branch_id"],
    "branches_clean": ["branch_id"],
    "tickets_clean": ["ticket_id", "request_id", "customer_id"],
    "updates_clean": ["update_id", "request_id"],
    "complaints_clean": ["complaint_id", "request_id", "customer_id"]
}

clean_tables = {
    "customers_clean": customers_clean,
    "branches_clean": branches_clean,
    "tickets_clean": tickets_clean,
    "updates_clean": updates_clean,
    "complaints_clean": complaints_clean
}

for table_name, columns in identifier_columns.items():
    df = clean_tables[table_name]

    for column in columns:
        if column in df.columns:
            df[column] = (
                df[column]
                .astype("string")
                .str.strip()
                .replace("", pd.NA)
            )

primary_keys = {
    "customers_clean": "customer_id",
    "branches_clean": "branch_id",
    "tickets_clean": "ticket_id",
    "updates_clean": "update_id",
    "complaints_clean": "complaint_id"
}

duplicate_audit = []

for table_name, primary_key in primary_keys.items():
    df = clean_tables[table_name]

    if primary_key not in df.columns:
        raise KeyError(
            f"{primary_key} is missing from {table_name}"
        )

    duplicate_mask = df[primary_key].notna() & (
        df[primary_key].duplicated(keep="first")
    )

    duplicate_count = int(duplicate_mask.sum())

    df = df.loc[~duplicate_mask].copy()

    clean_tables[table_name] = df

    duplicate_audit.append({
        "dataset": table_name,
        "primary_key": primary_key,
        "duplicate_records_removed": duplicate_count,
        "missing_primary_keys": int(df[primary_key].isna().sum()),
        "records_after_deduplication": len(df)
    })

customers_clean = clean_tables["customers_clean"]
branches_clean = clean_tables["branches_clean"]
tickets_clean = clean_tables["tickets_clean"]
updates_clean = clean_tables["updates_clean"]
complaints_clean = clean_tables["complaints_clean"]

duplicate_audit_df = pd.DataFrame(duplicate_audit)

print("PRIMARY KEY DEDUPLICATION AUDIT")
display(duplicate_audit_df)

def clean_text_columns(df, columns):
    for column in columns:
        if column in df.columns:
            df[column] = (
                df[column]
                .astype("string")
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
    return df


customers_clean = clean_text_columns(
    customers_clean,
    ["customer_name", "customer_segment", "account_type"]
)

branches_clean = clean_text_columns(
    branches_clean,
    [
        "branch_name",
        "branch_city",
        "branch_state",
        "branch_zone",
        "branch_type"
    ]
)

tickets_clean = clean_text_columns(
    tickets_clean,
    ["ticket_status", "assigned_team"]
)

updates_clean = clean_text_columns(
    updates_clean,
    [
        "status_from",
        "status_to",
        "updated_by_role",
        "update_note"
    ]
)

complaints_clean = clean_text_columns(
    complaints_clean,
    [
        "complaint_type",
        "complaint_description",
        "complaint_status",
        "sentiment_tag"
    ]
)

for df, column in [
    (tickets_clean, "ticket_status"),
    (complaints_clean, "complaint_status"),
    (updates_clean, "status_from"),
    (updates_clean, "status_to")
]:
    if column in df.columns:
        df[column] = (
            df[column]
            .str.lower()
            .str.replace(r"[\s-]+", "_", regex=True)
        )

if "customer_segment" in customers_clean.columns:
    customers_clean["customer_segment"] = (
        customers_clean["customer_segment"].str.title()
    )


if "sentiment_tag" in complaints_clean.columns:
    complaints_clean["sentiment_tag"] = (
        complaints_clean["sentiment_tag"].str.lower()
    )

def normalize_flag(series):
    """
    Convert common true/false representations into nullable booleans.
    Unknown values remain missing rather than being assumed False.
    """
    normalized = series.astype("string").str.strip().str.lower()

    true_values = {"true", "1", "yes", "y", "t"}
    false_values = {"false", "0", "no", "n", "f"}

    result = pd.Series(
        pd.NA,
        index=series.index,
        dtype="boolean"
    )

    result.loc[normalized.isin(true_values)] = True
    result.loc[normalized.isin(false_values)] = False

    return result


for df, column in [
    (tickets_clean, "escalation_flag"),
    (updates_clean, "reassignment_flag")
]:
    if column in df.columns:
        df[column] = normalize_flag(df[column])

date_columns = {
    "tickets_clean": ["ticket_created_date"],
    "updates_clean": ["update_date"],
    "complaints_clean": ["complaint_date"]
}

for table_name, columns in date_columns.items():
    df = clean_tables[table_name]

    for column in columns:
        if column in df.columns:
            df[column] = pd.to_datetime(
                df[column],
                errors="coerce"
            )

    clean_tables[table_name] = df

tickets_clean = clean_tables["tickets_clean"]
updates_clean = clean_tables["updates_clean"]
complaints_clean = clean_tables["complaints_clean"]

if "resolution_time_hours" in tickets_clean.columns:
    tickets_clean["resolution_time_hours"] = pd.to_numeric(
        tickets_clean["resolution_time_hours"],
        errors="coerce"
    )

if "relationship_age_days" in customers_clean.columns:
    customers_clean["relationship_age_days"] = pd.to_numeric(
        customers_clean["relationship_age_days"],
        errors="coerce"
    )

valid_request_ids = set(
    requests_clean["request_id"].dropna().astype(str).str.strip()
)

valid_customer_ids = set(
    customers_clean["customer_id"].dropna().astype(str).str.strip()
)

valid_branch_ids = set(
    branches_clean["branch_id"].dropna().astype(str).str.strip()
)


def count_unmatched(df, column, valid_ids):
    if column not in df.columns:
        return np.nan

    values = df[column].astype("string").str.strip()

    return int(
        (
            values.notna()
            & ~values.isin(valid_ids)
        ).sum()
    )


relationship_audit = []

for table_name, df in [
    ("tickets_clean", tickets_clean),
    ("updates_clean", updates_clean),
    ("complaints_clean", complaints_clean)
]:
    relationship_audit.append({
        "dataset": table_name,
        "unmatched_request_ids": count_unmatched(
            df, "request_id", valid_request_ids
        ),
        "unmatched_customer_ids": count_unmatched(
            df, "customer_id", valid_customer_ids
        )
    })

relationship_audit.append({
    "dataset": "customers_clean",
    "unmatched_request_ids": np.nan,
    "unmatched_customer_ids": np.nan
})

relationship_audit_df = pd.DataFrame(relationship_audit)

print("\nFOREIGN KEY VALIDATION")
display(relationship_audit_df)

if "preferred_branch_id" in customers_clean.columns:
    preferred_branch_ids = (
        customers_clean["preferred_branch_id"]
        .astype("string")
        .str.strip()
    )

    unmatched_preferred_branches = int(
        (
            preferred_branch_ids.notna()
            & ~preferred_branch_ids.isin(valid_branch_ids)
        ).sum()
    )

    print(
        "Customers with unmatched preferred branches:",
        unmatched_preferred_branches
    )


if "branch_id" in requests_clean.columns:
    request_branch_ids = (
        requests_clean["branch_id"]
        .astype("string")
        .str.strip()
    )

    unmatched_request_branches = int(
        (
            request_branch_ids.notna()
            & ~request_branch_ids.isin(valid_branch_ids)
        ).sum()
    )

    print(
        "Service requests with unmatched branch IDs:",
        unmatched_request_branches
    )

final_tables = {
    "customers_clean": customers_clean,
    "branches_clean": branches_clean,
    "tickets_clean": tickets_clean,
    "updates_clean": updates_clean,
    "complaints_clean": complaints_clean
}

count_reconciliation = []

for table_name, df in final_tables.items():
    count_reconciliation.append({
        "dataset": table_name,
        "original_rows": original_counts[table_name.replace(
            "_clean", ""
        )],
        "retained_rows": len(df),
        "distinct_primary_keys": int(
            df[primary_keys[table_name]].nunique()
        ),
        "missing_primary_keys": int(
            df[primary_keys[table_name]].isna().sum()
        )
    })

count_reconciliation_df = pd.DataFrame(count_reconciliation)

print("\nRECORD COUNT RECONCILIATION")
display(count_reconciliation_df)

print("\nTICKET STATUS DISTRIBUTION")
display(
    tickets_clean["ticket_status"]
    .value_counts(dropna=False)
    .rename_axis("ticket_status")
    .reset_index(name="count")
)

print("\nCOMPLAINT STATUS DISTRIBUTION")
display(
    complaints_clean["complaint_status"]
    .value_counts(dropna=False)
    .rename_axis("complaint_status")
    .reset_index(name="count")
)

print("\nUPDATE STATUS DISTRIBUTION")
display(
    updates_clean["status_to"]
    .value_counts(dropna=False)
    .rename_axis("status_to")
    .reset_index(name="count")
)


print("MILESTONE 04 COMPLETED")


for name, df in final_tables.items():
    print(f"{name}: {len(df)} rows, {len(df.columns)} columns")

print("\nCleaned event previews:")

display(tickets_clean.head(5))
display(updates_clean.head(5))
display(complaints_clean.head(5))

for table_name, primary_key in primary_keys.items():
    df = final_tables[table_name]

    assert not df[primary_key].dropna().duplicated().any(), (
        f"Duplicate {primary_key} values remain in {table_name}"
    )

print(
    "\nMilestone 04 checks passed: "
    "nonmissing primary keys are unique, and event rows "
    "were not deduplicated using request_id."
)

PRIMARY KEY DEDUPLICATION AUDIT


,dataset,primary_key,duplicate_records_removed,missing_primary_keys,records_after_deduplication
0,customers_clean,customer_id,0,0,1200
1,branches_clean,branch_id,0,0,1000
2,tickets_clean,ticket_id,0,0,1500
3,updates_clean,update_id,0,0,1300
4,complaints_clean,complaint_id,0,0,1500



FOREIGN KEY VALIDATION


,dataset,unmatched_request_ids,unmatched_customer_ids
0,tickets_clean,31.0,24.0
1,updates_clean,30.0,NaN
2,complaints_clean,29.0,21.0
3,customers_clean,NaN,NaN


Customers with unmatched preferred branches: 0
Service requests with unmatched branch IDs: 11

RECORD COUNT RECONCILIATION


,dataset,original_rows,retained_rows,distinct_primary_keys,missing_primary_keys
0,customers_clean,1200,1200,1200,0
1,branches_clean,1000,1000,1000,0
2,tickets_clean,1500,1500,1500,0
3,updates_clean,1300,1300,1300,0
4,complaints_clean,1500,1500,1500,0


... (7 outputs left out)

,ticket_id,request_id,customer_id,ticket_created_date,ticket_status,assigned_team,resolution_time_hours,escalation_flag
0,TKT000001,REQ0001676,CUST000879,2026-02-10,pending,Digital Banking,148.0,False
1,TKT000002,REQ0000292,CUST000404,2026-05-27,closed,Digital Banking,56.0,True
2,TKT000003,REQ0003061,CUST000604,2026-02-06,resolved,Digital Banking,NaN,False
3,TKT000004,REQ0002708,CUST000126,2026-04-12,escalated,Grievance Desk,157.0,True
4,TKT000005,REQ0003375,CUST000928,2026-05-15,reopened,Digital Banking,186.0,False


,update_id,request_id,update_date,status_from,status_to,updated_by_role,reassignment_flag,update_note
0,UPD000001,REQ0004710,2026-02-09,pending_customer,resolved,Branch Ops,False,Backend team asked for more details.
1,UPD000002,REQ0001800,2026-05-31,escalated,pending_customer,Digital Banking,False,Marked resolved after confirmation.
2,UPD000003,REQ0003051,2026-01-23,escalated,closed,Cards Ops,True,Waiting for branch verification.
3,UPD000004,REQ0002576,2026-01-31,open,pending_customer,Grievance Desk,True,Waiting for branch verification.
4,UPD000005,REQ0000561,2026-03-27,in_progress,closed,Branch Ops,False,Marked resolved after confirmation.


,complaint_id,request_id,customer_id,complaint_date,complaint_type,complaint_description,complaint_status,sentiment_tag
0,CMP000001,REQ0000649,CUST000908,2026-05-22,KYC Delay,Customer says documents were submitted but req...,open,negative
1,CMP000002,REQ0002883,CUST000578,2026-01-24,Loan Servicing,Customer received conflicting updates from bra...,in_review,negative
2,CMP000003,REQ0004858,CUST000734,2026-01-31,Loan Servicing,Customer says request has not moved despite mu...,closed,negative
3,CMP000004,REQ0004390,CUST000888,2026-02-22,KYC Delay,Customer requests supervisor callback and writ...,closed,angry
4,CMP000005,REQ0001781,CUST000357,2026-03-25,Digital Login Issue,Customer claims commitment date was missed.,open,negative



Milestone 04 checks passed: nonmissing primary keys are unique, and event rows were not deduplicated using request_id.


## Milestone 05 · Measure SLA reliability

**From the team:** Management needs reproducible breached and still-pending cases, not artificial breaches caused by using a later system clock.

**Your responsibility:** Use an explicit reporting snapshot derived from the source dates, calculate elapsed hours from request creation through resolution or snapshot, flag completed vs open breaches, and summarize category trends.

**Handover evidence:** `sla_df` with `elapsed_hours`, `sla_breached`, `open_sla_breach`, `sla_date_issue`, plus `category_sla_summary`.

In [7]:
import os
import glob
import json
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display

def find_file(filename):
    paths = (
        glob.glob(f"/content/**/{filename}", recursive=True)
        + glob.glob(f"/mnt/data/**/{filename}", recursive=True)
        + glob.glob(f"./**/{filename}", recursive=True)
    )
    return paths[0] if paths else None

service_requests_raw = globals().get("service_requests_raw")
request_status_updates_raw = globals().get("request_status_updates_raw")
customer_complaints_raw = globals().get("customer_complaints_raw")
support_tickets_raw = globals().get("support_tickets_raw")
branches_raw = globals().get("branches_raw")

dataset_zip = None

if not isinstance(service_requests_raw, pd.DataFrame):
    csv_path = find_file("service_requests.csv")

    if csv_path:
        service_requests_raw = pd.read_csv(csv_path)
    else:
        zip_candidates = (
            glob.glob("/content/*.zip")
            + glob.glob("/mnt/data/*.zip")
            + glob.glob("./*.zip")
        )

        for path in zip_candidates:
            try:
                with zipfile.ZipFile(path, "r") as zf:
                    if "service_requests.csv" in zf.namelist():
                        dataset_zip = path
                        break
            except zipfile.BadZipFile:
                continue

        if dataset_zip is None:
            raise FileNotFoundError(
                "Dataset not found. Upload the ZIP containing "
                "service_requests.csv to Google Colab and run again."
            )

        with zipfile.ZipFile(dataset_zip, "r") as zf:
            service_requests_raw = pd.read_csv(
                zf.open("service_requests.csv")
            )

            if "request_status_updates.xlsx" in zf.namelist():
                request_status_updates_raw = pd.read_excel(
                    zf.open("request_status_updates.xlsx"),
                    sheet_name="request_status_updates"
                )

            if "service_evidence.json" in zf.namelist():
                evidence = json.load(zf.open("service_evidence.json"))

                if isinstance(evidence, dict):
                    if not isinstance(customer_complaints_raw, pd.DataFrame):
                        customer_complaints_raw = pd.DataFrame(
                            evidence.get("customer_complaints", [])
                        )

                    if not isinstance(support_tickets_raw, pd.DataFrame):
                        support_tickets_raw = pd.DataFrame(
                            evidence.get("support_tickets", [])
                        )

                    if not isinstance(branches_raw, pd.DataFrame):
                        branches_raw = pd.DataFrame(
                            evidence.get("branches", [])
                        )

if not isinstance(request_status_updates_raw, pd.DataFrame):
    path = find_file("request_status_updates.xlsx")

    if path:
        request_status_updates_raw = pd.read_excel(
            path,
            sheet_name="request_status_updates"
        )
    else:
        request_status_updates_raw = pd.DataFrame()

if not isinstance(customer_complaints_raw, pd.DataFrame):
    customer_complaints_raw = pd.DataFrame()

if not isinstance(support_tickets_raw, pd.DataFrame):
    support_tickets_raw = pd.DataFrame()

if not isinstance(branches_raw, pd.DataFrame):
    branches_raw = pd.DataFrame()

raw = service_requests_raw.copy()

required_columns = [
    "request_id",
    "request_created_date",
    "request_created_time",
    "request_category",
    "request_status",
    "sla_hours",
    "resolved_date"
]

missing = [col for col in required_columns if col not in raw.columns]

if missing:
    raise ValueError(f"Missing columns in source dataset: {missing}")

created_date = pd.to_datetime(
    raw["request_created_date"], errors="coerce"
)

created_time = pd.to_timedelta(
    raw["request_created_time"].fillna("00:00:00").astype(str),
    errors="coerce"
)

raw["request_created_ts"] = created_date + created_time

raw["resolved_date_parsed"] = pd.to_datetime(
    raw["resolved_date"], errors="coerce"
)

raw["effective_sla_hours"] = pd.to_numeric(
    raw["sla_hours"], errors="coerce"
)


raw = raw.drop_duplicates(
    subset=["request_id"], keep="last"
).copy()

requests = raw.copy()

source_date_series = []

def collect_source_dates(dataframe, date_columns):
    if isinstance(dataframe, pd.DataFrame) and not dataframe.empty:
        for column in date_columns:
            if column in dataframe.columns:
                dates = pd.to_datetime(
                    dataframe[column], errors="coerce"
                ).dropna()

                if not dates.empty:
                    source_date_series.append(dates)

collect_source_dates(
    service_requests_raw,
    [
        "request_created_date",
        "last_updated_date",
        "resolved_date"
    ]
)

collect_source_dates(
    request_status_updates_raw,
    ["update_date"]
)

collect_source_dates(
    customer_complaints_raw,
    [
        "complaint_date",
        "created_at",
        "created_date",
        "last_updated_date"
    ]
)

collect_source_dates(
    support_tickets_raw,
    [
        "ticket_created_date",
        "created_at",
        "created_date",
        "last_updated_date"
    ]
)

if not source_date_series:
    raise ValueError(
        "Cannot calculate snapshot because no valid source dates were found."
    )

snapshot_date = max(
    dates.max() for dates in source_date_series
).normalize()

snapshot_ts = (
    snapshot_date
    + pd.Timedelta(days=1)
    - pd.Timedelta(seconds=1)
)

print("Reporting snapshot date:", snapshot_date.date())
print("Snapshot timestamp:", snapshot_ts)

status_text = (
    requests["request_status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

completed_statuses = {
    "resolved",
    "closed",
    "completed",
    "complete",
    "fulfilled"
}

requests["is_completed"] = (
    status_text.isin(completed_statuses)
    | requests["resolved_date_parsed"].notna()
)

requests["is_open"] = ~requests["is_completed"]

sla_df = requests.copy()

sla_df["sla_end_ts"] = sla_df["resolved_date_parsed"].where(
    sla_df["resolved_date_parsed"].notna(),
    snapshot_ts
)

sla_df["elapsed_hours"] = (
    sla_df["sla_end_ts"] - sla_df["request_created_ts"]
).dt.total_seconds() / 3600

invalid_resolution_order = (
    sla_df["resolved_date_parsed"].notna()
    & sla_df["request_created_ts"].notna()
    & (
        sla_df["resolved_date_parsed"]
        < sla_df["request_created_ts"]
    )
)

sla_df["sla_date_issue"] = (
    sla_df["request_created_ts"].isna()
    | invalid_resolution_order
    | (sla_df["elapsed_hours"] < 0)
)

sla_df.loc[
    sla_df["sla_date_issue"],
    "elapsed_hours"
] = np.nan

sla_df["sla_value_issue"] = (
    sla_df["effective_sla_hours"].isna()
    | (sla_df["effective_sla_hours"] <= 0)
)

sla_df["sla_breached"] = (
    ~sla_df["sla_date_issue"]
    & ~sla_df["sla_value_issue"]
    & sla_df["elapsed_hours"].notna()
    & (
        sla_df["elapsed_hours"]
        > sla_df["effective_sla_hours"]
    )
)

sla_df["open_sla_breach"] = (
    sla_df["is_open"]
    & sla_df["sla_breached"]
)

sla_df["completed_sla_breach"] = (
    sla_df["is_completed"]
    & sla_df["sla_breached"]
)

sla_df["sla_remaining_hours"] = (
    sla_df["effective_sla_hours"]
    - sla_df["elapsed_hours"]
)

sla_df["approaching_sla_breach"] = (
    sla_df["is_open"]
    & ~sla_df["sla_breached"]
    & sla_df["sla_remaining_hours"].between(
        0, 12, inclusive="both"
    )
)

sla_df["request_category"] = (
    sla_df["request_category"].fillna("Unknown")
)

category_sla_summary = (
    sla_df.groupby("request_category", dropna=False)
    .agg(
        total_requests=("request_id", "nunique"),
        completed_requests=("is_completed", "sum"),
        open_requests=("is_open", "sum"),
        sla_date_issues=("sla_date_issue", "sum"),
        sla_value_issues=("sla_value_issue", "sum"),
        breached_requests=("sla_breached", "sum"),
        completed_sla_breaches=("completed_sla_breach", "sum"),
        open_sla_breaches=("open_sla_breach", "sum"),
        approaching_breaches=("approaching_sla_breach", "sum"),
        avg_elapsed_hours=("elapsed_hours", "mean")
    )
    .reset_index()
)

valid_records = sla_df.loc[
    ~sla_df["sla_date_issue"]
    & ~sla_df["sla_value_issue"]
].copy()

valid_counts = (
    valid_records.groupby("request_category", dropna=False)
    .agg(valid_sla_requests=("request_id", "nunique"))
    .reset_index()
)

category_sla_summary = category_sla_summary.merge(
    valid_counts,
    on="request_category",
    how="left"
)

category_sla_summary["valid_sla_requests"] = (
    category_sla_summary["valid_sla_requests"]
    .fillna(0)
    .astype(int)
)

category_sla_summary["breach_rate_pct"] = np.where(
    category_sla_summary["valid_sla_requests"] > 0,
    (
        category_sla_summary["breached_requests"]
        / category_sla_summary["valid_sla_requests"]
    ) * 100,
    np.nan
)

category_sla_summary["sla_breaches"] = (
    category_sla_summary["breached_requests"]
)

category_sla_summary["unresolved_requests"] = (
    category_sla_summary["open_requests"]
)

category_sla_summary = category_sla_summary.sort_values(
    ["open_sla_breaches", "breached_requests"],
    ascending=False
).reset_index(drop=True)

branch_sla_summary = pd.DataFrame()

if (
    not branches_raw.empty
    and "branch_id" in sla_df.columns
    and "branch_id" in branches_raw.columns
):
    branch_data = sla_df.merge(
        branches_raw.drop_duplicates("branch_id"),
        on="branch_id",
        how="left"
    )

    group_columns = ["branch_id"]

    for col in [
        "branch_name",
        "branch_city",
        "branch_state",
        "branch_zone"
    ]:
        if col in branch_data.columns:
            group_columns.append(col)

    branch_sla_summary = (
        branch_data.groupby(group_columns, dropna=False)
        .agg(
            total_requests=("request_id", "nunique"),
            unresolved_requests=("is_open", "sum"),
            sla_breaches=("sla_breached", "sum"),
            open_sla_breaches=("open_sla_breach", "sum"),
            completed_sla_breaches=("completed_sla_breach", "sum"),
            sla_date_issues=("sla_date_issue", "sum")
        )
        .reset_index()
        .sort_values(
            ["open_sla_breaches", "sla_breaches"],
            ascending=False
        )
        .reset_index(drop=True)
    )

sla_output_columns = [
    "elapsed_hours",
    "sla_breached",
    "open_sla_breach",
    "completed_sla_breach",
    "sla_date_issue",
    "sla_value_issue",
    "sla_remaining_hours",
    "approaching_sla_breach",
    "is_open",
    "is_completed"
]

requests = requests.drop(
    columns=[
        col for col in sla_output_columns
        if col in requests.columns
    ],
    errors="ignore"
).merge(
    sla_df[["request_id"] + sla_output_columns],
    on="request_id",
    how="left",
    validate="one_to_one"
)

requests["is_unresolved"] = requests["is_open"]


print("\n      SLA RELIABILITY SUMMARY    ")
print("Rows in sla_df:", len(sla_df))
print("Unique requests:", sla_df["request_id"].nunique())
print("Completed requests:", int(sla_df["is_completed"].sum()))
print("Open requests:", int(sla_df["is_open"].sum()))
print("Open SLA breaches:", int(sla_df["open_sla_breach"].sum()))
print("Completed SLA breaches:", int(sla_df["completed_sla_breach"].sum()))
print("Total SLA breaches:", int(sla_df["sla_breached"].sum()))
print("Date issues:", int(sla_df["sla_date_issue"].sum()))
print("SLA value issues:", int(sla_df["sla_value_issue"].sum()))

print("\n    CATEGORY SLA SUMMARY    ")
display(category_sla_summary)

print("\n   SLA DATA PREVIEW   ")
display(
    sla_df[
        [
            "request_id",
            "request_created_ts",
            "resolved_date_parsed",
            "request_category",
            "request_status",
            "effective_sla_hours",
            "elapsed_hours",
            "sla_breached",
            "open_sla_breach",
            "sla_date_issue"
        ]
    ].head(10)
)

required_outputs = [
    "elapsed_hours",
    "sla_breached",
    "open_sla_breach",
    "sla_date_issue"
]

assert all(col in sla_df.columns for col in required_outputs)
assert isinstance(category_sla_summary, pd.DataFrame)

print("\nMilestone 05 completed successfully.")
print("Created: sla_df, category_sla_summary, branch_sla_summary")

Reporting snapshot date: 2026-07-01
Snapshot timestamp: 2026-07-01 23:59:59

      SLA RELIABILITY SUMMARY    
Rows in sla_df: 4992
Unique requests: 4992
Completed requests: 2362
Open requests: 2630
Open SLA breaches: 2629
Completed SLA breaches: 729
Total SLA breaches: 3358
Date issues: 287
SLA value issues: 0

    CATEGORY SLA SUMMARY    


,request_category,total_requests,completed_requests,open_requests,sla_date_issues,sla_value_issues,breached_requests,completed_sla_breaches,open_sla_breaches,approaching_breaches,avg_elapsed_hours,valid_sla_requests,breach_rate_pct,sla_breaches,unresolved_requests
0,Digital Banking,821,363,458,87,0,534,76,458,0,1475.602779,734,72.752044,534,458
1,Account,843,392,451,47,0,573,122,451,0,1335.521597,796,71.984925,573,451
2,Card,806,379,427,58,0,532,105,427,0,1378.744672,748,71.122995,532,427
3,Loan,806,386,420,28,0,564,145,419,0,1228.712425,778,72.493573,564,420
4,Complaint,803,400,403,30,0,536,133,403,0,1232.354081,773,69.340233,536,403
5,KYC,779,376,403,29,0,529,126,403,0,1289.279050,750,70.533333,529,403
6,account,23,7,16,0,0,21,5,16,0,1584.506329,23,91.304348,21,16
7,kyc,23,10,13,1,0,16,3,13,0,1471.062715,22,72.727273,16,13
8,complaint,19,8,11,0,0,13,2,11,0,1512.451594,19,68.421053,13,11
9,loan,30,20,10,2,0,15,5,10,0,718.607044,28,53.571429,15,10



   SLA DATA PREVIEW   


,request_id,request_created_ts,resolved_date_parsed,request_category,request_status,effective_sla_hours,elapsed_hours,sla_breached,open_sla_breach,sla_date_issue
0,REQ0000009,2026-04-27 04:20:00,NaT,Complaint,In Progress,72,1579.666389,True,True,False
1,REQ0000010,2026-05-17 22:31:00,NaT,Card,Pending Customer,36,1081.483056,True,True,False
2,REQ0000011,2026-06-19 04:47:00,2026-06-19,Digital Banking,Closed,24,NaN,False,False,True
3,REQ0000012,2026-04-20 03:36:00,NaT,Complaint,In Progress,72,1748.399722,True,True,False
4,REQ0000013,2026-06-13 07:18:00,2026-06-16,Account,Closed,48,64.700000,True,False,False
5,REQ0000014,2026-04-25 06:03:00,2026-04-26,KYC,Closed,72,17.950000,False,False,False
6,REQ0000015,2026-02-22 19:15:00,2026-02-24,Account,resolved,48,28.750000,False,False,False
7,REQ0000016,2026-05-25 15:46:00,2026-05-27,Account,Resolved,48,32.233333,False,False,False
8,REQ0000017,2026-04-21 18:52:00,NaT,digital banking,Pending Customer,24,1709.133056,True,True,False
9,REQ0000018,2026-04-24 02:08:00,2026-04-28,Complaint,Resolved,72,93.866667,True,False,False



Milestone 05 completed successfully.
Created: sla_df, category_sla_summary, branch_sla_summary


## Milestone 06 · Analyze branch ownership and customer segments

**From the team:** Branches need their workloads and breach ratios, including requests linked to missing branch metadata.

**Your responsibility:** Join branch/customer attributes with many-to-one validation, analyze city/zone/segment and create branch metrics.

**Handover evidence:** `requests_enriched` retaining all requests; `branch_sla_summary` with total requests, breached count and breach rate.

In [8]:
import pandas as pd
import numpy as np
from IPython.display import display

print("MILESTONE 06: BRANCH OWNERSHIP AND CUSTOMER SEGMENT ANALYSIS")

required_dataframes = [
    "requests_clean",
    "customers_clean",
    "branches_clean",
    "sla_df"
]

for name in required_dataframes:
    if name not in globals():
        raise NameError(
            f"{name} not found. Please run the earlier milestones first."
        )

requests_base = requests_clean.copy()
customers_dim = customers_clean.copy()
branches_dim = branches_clean.copy()
sla_source = sla_df.copy()

if not {"request_id", "customer_id", "branch_id"}.issubset(
    requests_base.columns
):
    raise ValueError(
        "requests_clean must contain request_id, customer_id and branch_id."
    )

if not {"customer_id"}.issubset(customers_dim.columns):
    raise ValueError("customers_clean must contain customer_id.")

if not {"branch_id"}.issubset(branches_dim.columns):
    raise ValueError("branches_clean must contain branch_id.")

if not {"request_id", "sla_breached"}.issubset(sla_source.columns):
    raise ValueError(
        "sla_df must contain request_id and sla_breached."
    )

if requests_base["request_id"].isna().any():
    raise ValueError("requests_clean contains missing request_id values.")

if not requests_base["request_id"].is_unique:
    raise ValueError(
        "requests_clean must contain exactly one row per request_id."
    )

for frame, key, name in [
    (customers_dim, "customer_id", "customers_clean"),
    (branches_dim, "branch_id", "branches_clean")
]:
    if frame[key].isna().any():
        raise ValueError(
            f"{name} contains missing {key} values. Clean dimension keys first."
        )

    if not frame[key].is_unique:
        raise ValueError(
            f"{name} contains duplicate {key} values. "
            "Resolve duplicates before merging."
        )

if sla_source["request_id"].duplicated().any():
    raise ValueError(
        "sla_df has duplicate request_id values. "
        "Resolve them before merging."
    )

print(f"\nRequests before enrichment: {len(requests_base):,}")

customer_columns = [
    column for column in customers_dim.columns
    if column == "customer_id" or column not in requests_base.columns
]

customer_lookup = customers_dim[customer_columns].copy()

requests_enriched = requests_base.merge(
    customer_lookup,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator="_customer_merge"
)

requests_enriched["customer_metadata_missing"] = (
    requests_enriched["_customer_merge"] == "left_only"
)

requests_enriched.drop(
    columns=["_customer_merge"],
    inplace=True
)

branch_columns = [
    column for column in branches_dim.columns
    if column == "branch_id" or column not in requests_enriched.columns
]

branch_lookup = branches_dim[branch_columns].copy()

requests_enriched = requests_enriched.merge(
    branch_lookup,
    on="branch_id",
    how="left",
    validate="many_to_one",
    indicator="_branch_merge"
)

requests_enriched["branch_metadata_missing"] = (
    requests_enriched["_branch_merge"] == "left_only"
)

requests_enriched.drop(
    columns=["_branch_merge"],
    inplace=True
)

requests_enriched["branch_metadata_missing"] = (
    requests_enriched["branch_metadata_missing"]
    | requests_enriched["branch_id"].isna()
)


sla_columns = [
    column for column in sla_source.columns
    if column == "request_id" or column not in requests_enriched.columns
]

sla_lookup = sla_source[sla_columns].copy()

requests_enriched = requests_enriched.merge(
    sla_lookup,
    on="request_id",
    how="left",
    validate="one_to_one"
)

if len(requests_enriched) != len(requests_base):
    raise AssertionError(
        "Request count changed during enrichment."
    )

if not requests_enriched["request_id"].is_unique:
    raise AssertionError(
        "Duplicate request IDs were created during enrichment."
    )

if set(requests_enriched["request_id"]) != set(
    requests_base["request_id"]
):
    raise AssertionError(
        "Some request IDs were lost or added during enrichment."
    )

print("\nCustomer and branch attributes merged successfully.")
print("Join validation: many-to-one")
print(f"Requests after enrichment: {len(requests_enriched):,}")
print("All original request IDs preserved: YES")

def convert_to_boolean(value):
    if pd.isna(value):
        return pd.NA

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    normalized = str(value).strip().lower()

    if normalized in ["true", "1", "yes", "y"]:
        return True

    if normalized in ["false", "0", "no", "n"]:
        return False

    return pd.NA


requests_enriched["sla_breached"] = (
    requests_enriched["sla_breached"]
    .apply(convert_to_boolean)
    .astype("boolean")
)

def find_column(frame, candidates):
    for column in candidates:
        if column in frame.columns:
            return column
    return None


branch_name_col = find_column(
    requests_enriched,
    ["branch_name", "branch", "branch_code"]
)

city_col = find_column(
    requests_enriched,
    ["city", "branch_city", "city_name"]
)

zone_col = find_column(
    requests_enriched,
    ["zone", "branch_zone", "region", "region_name"]
)

segment_col = find_column(
    requests_enriched,
    ["customer_segment", "segment", "customer_type", "segment_name"]
)

requests_enriched["branch_reporting_name"] = (
    requests_enriched[branch_name_col].astype("string")
    if branch_name_col
    else pd.Series(pd.NA, index=requests_enriched.index, dtype="string")
)

requests_enriched["branch_reporting_name"] = (
    requests_enriched["branch_reporting_name"]
    .fillna(requests_enriched["branch_id"].astype("string"))
    .fillna("Missing Branch ID")
)

requests_enriched.loc[
    requests_enriched["branch_metadata_missing"],
    "branch_reporting_name"
] = "Missing Branch Metadata"

if city_col:
    requests_enriched["city_reporting"] = (
        requests_enriched[city_col]
        .astype("string")
        .fillna("Unknown City")
    )
else:
    requests_enriched["city_reporting"] = "Unknown City"

if zone_col:
    requests_enriched["zone_reporting"] = (
        requests_enriched[zone_col]
        .astype("string")
        .fillna("Unknown Zone")
    )
else:
    requests_enriched["zone_reporting"] = "Unknown Zone"

if segment_col:
    requests_enriched["segment_reporting"] = (
        requests_enriched[segment_col]
        .astype("string")
        .fillna("Unknown Segment")
    )
else:
    requests_enriched["segment_reporting"] = "Unknown Segment"

branch_group_columns = [
    "branch_id",
    "branch_reporting_name",
    "city_reporting",
    "zone_reporting"
]

branch_sla_summary = (
    requests_enriched
    .groupby(
        branch_group_columns,
        dropna=False
    )
    .agg(
        total_requests=("request_id", "nunique"),
        breached_requests=(
            "sla_breached",
            lambda values: int((values == True).sum())
        ),
        known_sla_requests=(
            "sla_breached",
            lambda values: int(values.notna().sum())
        ),
        missing_branch_metadata=(
            "branch_metadata_missing",
            "max"
        )
    )
    .reset_index()
)

branch_sla_summary["sla_breached_requests"] = (
    branch_sla_summary["breached_requests"]
)

branch_sla_summary["breach_rate"] = np.where(
    branch_sla_summary["known_sla_requests"] > 0,
    (
        branch_sla_summary["breached_requests"]
        / branch_sla_summary["known_sla_requests"]
    ),
    np.nan
)

segment_sla_summary = (
    requests_enriched
    .groupby("segment_reporting", dropna=False)
    .agg(
        total_requests=("request_id", "nunique"),
        breached_requests=(
            "sla_breached",
            lambda values: int((values == True).sum())
        ),
        known_sla_requests=(
            "sla_breached",
            lambda values: int(values.notna().sum())
        )
    )
    .reset_index()
)

segment_sla_summary["breach_rate"] = np.where(
    segment_sla_summary["known_sla_requests"] > 0,
    (
        segment_sla_summary["breached_requests"]
        / segment_sla_summary["known_sla_requests"]
    ),
    np.nan
)


print("MILESTONE 06 VALIDATION")

assert len(requests_enriched) == len(requests_clean)

assert requests_enriched["request_id"].is_unique

assert set(requests_enriched["request_id"]) == set(
    requests_clean["request_id"]
)

assert {
    "total_requests",
    "breached_requests",
    "breach_rate"
}.issubset(branch_sla_summary.columns)

assert branch_sla_summary["total_requests"].sum() == len(
    requests_clean
)

valid_rates = branch_sla_summary["breach_rate"].dropna()

assert valid_rates.between(0, 1).all()

print("PASS: All requests retained.")
print("PASS: Request IDs remain unique.")
print("PASS: Customer and branch joins use many-to-one validation.")
print("PASS: Missing branch metadata is retained.")
print("PASS: Branch request counts reconcile with requests_clean.")
print("PASS: Breach rates are proportions between 0 and 1.")

print(
    "\nRequests with missing branch metadata:",
    int(requests_enriched["branch_metadata_missing"].sum())
)

print(
    "Requests with known SLA status:",
    int(requests_enriched["sla_breached"].notna().sum())
)

print(
    "Requests with an SLA breach:",
    int((requests_enriched["sla_breached"] == True).sum())
)


print("\n   Branch SLA Summary    ")
display(
    branch_sla_summary.sort_values(
        "total_requests",
        ascending=False
    ).reset_index(drop=True)
)

print("\n    Customer Segment SLA Summary   ")
display(
    segment_sla_summary.sort_values(
        "total_requests",
        ascending=False
    ).reset_index(drop=True)
)

print("\n    Enriched Requests Preview    ")
display(requests_enriched.head(10))

print("\nMilestone 06 completed successfully.")

MILESTONE 06: BRANCH OWNERSHIP AND CUSTOMER SEGMENT ANALYSIS

Requests before enrichment: 4,992

Customer and branch attributes merged successfully.
Join validation: many-to-one
Requests after enrichment: 4,992
All original request IDs preserved: YES
MILESTONE 06 VALIDATION
PASS: All requests retained.
PASS: Request IDs remain unique.
PASS: Customer and branch joins use many-to-one validation.
PASS: Missing branch metadata is retained.
PASS: Branch request counts reconcile with requests_clean.
PASS: Breach rates are proportions between 0 and 1.

Requests with missing branch metadata: 11
Requests with known SLA status: 4992
Requests with an SLA breach: 3358

   Branch SLA Summary    


,branch_id,branch_reporting_name,city_reporting,zone_reporting,total_requests,breached_requests,known_sla_requests,missing_branch_metadata,sla_breached_requests,breach_rate
0,BR0723,HDFC Surat Service Branch 0723,Surat,West,14,11,14,False,11,0.785714
1,BR0535,HDFC Ahmedabad Service Branch 0535,Ahmedabad,West,14,10,14,False,10,0.714286
2,BR0096,HDFC Madurai Service Branch 0096,Madurai,South,12,9,12,False,9,0.750000
3,BR0198,HDFC Salem Service Branch 0198,Salem,South,12,8,12,False,8,0.666667
4,BR0143,HDFC Gurugram Service Branch 0143,Gurugram,North,12,8,12,False,8,0.666667
...,...,...,...,...,...,...,...,...,...,...
1001,BRX0186,Missing Branch Metadata,Unknown City,Unknown Zone,1,1,1,True,1,1.000000
1002,BRX0187,Missing Branch Metadata,Unknown City,Unknown Zone,1,1,1,True,1,1.000000
1003,BRX0188,Missing Branch Metadata,Unknown City,Unknown Zone,1,0,1,True,0,0.000000
1004,BRX0189,Missing Branch Metadata,Unknown City,Unknown Zone,1,1,1,True,1,1.000000



    Customer Segment SLA Summary   


,segment_reporting,total_requests,breached_requests,known_sla_requests,breach_rate
0,Student,783,535,783,0.683269
1,Imperia,767,514,767,0.670143
2,Senior Citizen,721,464,721,0.643551
3,Preferred,709,476,709,0.671368
4,Nri,705,481,705,0.682270
5,Corporate,650,444,650,0.683077
6,Classic,643,435,643,0.676516
7,Unknown Segment,14,9,14,0.642857



    Enriched Requests Preview    


,request_id,customer_id,branch_id,request_created_date,request_created_time,request_category,request_subcategory,channel,request_status,sla_hours,...,sla_value_issue,sla_breached,open_sla_breach,completed_sla_breach,sla_remaining_hours,approaching_sla_breach,branch_reporting_name,city_reporting,zone_reporting,segment_reporting
0,REQ0000009,CUST001111,BR0566,2026-04-27,04:20:00,Complaint,Service Delay,Branch,in_progress,72,...,False,True,True,False,-1507.666389,False,HDFC Jodhpur Service Branch 0566,Jodhpur,North,Corporate
1,REQ0000010,CUST000649,BR0980,2026-05-17,22:31:00,Card,Failed ATM Withdrawal,Whatsapp,pending customer,36,...,False,True,True,False,-1045.483056,False,HDFC Kozhikode Service Branch 0980,Kozhikode,South,Corporate
2,REQ0000011,CUST000817,BR0045,2026-06-19,04:47:00,Digital Banking,NetBanking Login,Email,closed,24,...,False,False,False,False,NaN,False,HDFC Noida Service Branch 0045,Noida,North,Senior Citizen
3,REQ0000012,CUST000212,BR0335,2026-04-20,03:36:00,Complaint,Fraud Concern,Email,in_progress,72,...,False,True,True,False,-1676.399722,False,HDFC Hyderabad Service Branch 0335,Hyderabad,South,Nri
4,REQ0000013,CUST001137,BR0081,2026-06-13,07:18:00,Account,Balance Mismatch,Netbanking,closed,48,...,False,True,False,True,-16.700000,False,HDFC Kochi Service Branch 0081,Kochi,South,Corporate
5,REQ0000014,CUST000892,BR0091,2026-04-25,06:03:00,Kyc,Document Rejected,Call Center,closed,72,...,False,False,False,False,54.050000,False,HDFC Mysuru Service Branch 0091,Mysuru,South,Senior Citizen
6,REQ0000015,CUST001118,BR0262,2026-02-22,19:15:00,Account,Account Freeze,Email,resolved,48,...,False,False,False,False,19.250000,False,HDFC Coimbatore Service Branch 0262,Coimbatore,South,Preferred
7,REQ0000016,CUST000669,BR0422,2026-05-25,15:46:00,Account,Account Freeze,Branch,resolved,48,...,False,False,False,False,15.766667,False,HDFC Lucknow Service Branch 0422,Lucknow,North,Preferred
8,REQ0000017,CUST000492,BR0179,2026-04-21,18:52:00,Digital Banking,NetBanking Login,Whatsapp,pending customer,24,...,False,True,True,False,-1685.133056,False,HDFC Thiruvananthapuram Service Branch 0179,Thiruvananthapuram,South,Student
9,REQ0000018,CUST000133,BR0198,2026-04-24,02:08:00,Complaint,Fraud Concern,Whatsapp,resolved,72,...,False,True,False,True,-21.866667,False,HDFC Salem Service Branch 0198,Salem,South,Senior Citizen



Milestone 06 completed successfully.


## Milestone 07 · Investigate complaints, tickets and workflow updates

**From the team:** Escalations may be repeated across several events for a single request; supervisors need totals without duplicate request rows.

**Your responsibility:** Aggregate complaint severity/open status, ticket escalations and pending cases, and reassignment/status-update volumes at request level. Flag orphan events.

**Handover evidence:** `complaints_by_request`, `tickets_by_request`, `updates_by_request`, each unique on `request_id`; an orphan-link summary.

In [9]:
import pandas as pd
import numpy as np
from IPython.display import display

print("MILESTONE 07: COMPLAINTS, TICKETS & WORKFLOW ANALYSIS")

def get_source(clean_name, raw_name):
    if clean_name in globals() and isinstance(globals()[clean_name], pd.DataFrame):
        return globals()[clean_name].copy()

    if raw_name in globals() and isinstance(globals()[raw_name], pd.DataFrame):
        return globals()[raw_name].copy()

    raise NameError(
        f"Missing {clean_name} and {raw_name}. "
        "Run the earlier data-loading and cleaning milestones first."
    )


complaints_clean = get_source("complaints_clean", "complaints_df")
tickets_clean = get_source("tickets_clean", "tickets_df")
updates_clean = get_source("updates_clean", "updates_df")

if "requests_clean" not in globals():
    raise NameError("Run Milestone 03 first to create requests_clean.")

requests_clean = requests_clean.copy()

required_request_cols = {"request_id"}
if not required_request_cols.issubset(requests_clean.columns):
    raise KeyError("requests_clean must contain request_id.")


def normalize_id(series):
    result = series.astype("string").str.strip()
    return result.mask(result.eq(""))


def validate_event_source(df, event_id, source_name):
    required = [event_id, "request_id"]
    missing = [col for col in required if col not in df.columns]

    if missing:
        raise KeyError(
            f"{source_name} is missing required columns: {missing}"
        )

    df = df.copy()

    df[event_id] = normalize_id(df[event_id])
    df["request_id"] = normalize_id(df["request_id"])

    if df[event_id].isna().any():
        raise ValueError(
            f"{source_name} has missing {event_id} values. "
            "Review these source records before aggregating."
        )

    if df[event_id].duplicated().any():
        duplicate_count = int(df[event_id].duplicated().sum())
        raise ValueError(
            f"{source_name} has {duplicate_count} duplicate {event_id} values. "
            "Resolve duplicate event IDs before running the assessment."
        )

    return df


complaints_clean = validate_event_source(
    complaints_clean, "complaint_id", "complaints_clean"
)

tickets_clean = validate_event_source(
    tickets_clean, "ticket_id", "tickets_clean"
)

updates_clean = validate_event_source(
    updates_clean, "update_id", "updates_clean"
)

requests_clean["request_id"] = normalize_id(
    requests_clean["request_id"]
)

valid_request_ids = set(
    requests_clean.loc[
        requests_clean["request_id"].notna(), "request_id"
    ]
)

print("Service requests:", len(valid_request_ids))
print("Complaint events:", len(complaints_clean))
print("Support ticket events:", len(tickets_clean))
print("Status update events:", len(updates_clean))

if "complaint_status" not in complaints_clean.columns:
    raise KeyError("complaints_clean must contain complaint_status.")

complaints_clean["_status_normalized"] = (
    complaints_clean["complaint_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

open_complaint_statuses = {
    "open",
    "reopened",
    "in review",
    "pending",
    "in progress",
    "escalated",
    "unresolved"
}

complaints_clean["_is_open"] = (
    complaints_clean["_status_normalized"]
    .isin(open_complaint_statuses)
)

if "sentiment_tag" in complaints_clean.columns:
    complaints_clean["_is_negative_sentiment"] = (
        complaints_clean["sentiment_tag"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("negative")
    )
else:
    complaints_clean["_is_negative_sentiment"] = False

complaint_events = complaints_clean.loc[
    complaints_clean["request_id"].notna()
].copy()

complaints_by_request = (
    complaint_events.groupby("request_id")
    .agg(
        complaint_count=("complaint_id", "nunique"),
        open_complaint_count=("_is_open", "sum"),
        negative_sentiment_complaint_count=(
            "_is_negative_sentiment", "sum"
        )
    )
    .reset_index()
)

assert complaints_by_request["request_id"].is_unique

assert (
    complaints_by_request["complaint_count"].sum()
    == complaints_clean["request_id"].notna().sum()
)

if "ticket_status" not in tickets_clean.columns:
    raise KeyError("tickets_clean must contain ticket_status.")

tickets_clean["_status_normalized"] = (
    tickets_clean["ticket_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

pending_ticket_statuses = {
    "open",
    "pending",
    "reopened",
    "in progress",
    "pending customer",
    "awaiting customer",
    "waiting"
}

tickets_clean["_is_pending"] = (
    tickets_clean["_status_normalized"]
    .isin(pending_ticket_statuses)
)

if "escalation_flag" in tickets_clean.columns:
    tickets_clean["_is_escalated"] = (
        tickets_clean["escalation_flag"]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin({"yes", "true", "1", "y"})
    )
else:
  tickets_clean["_is_escalated"] = (
        tickets_clean["_status_normalized"] == "escalated"
    )

ticket_events = tickets_clean.loc[
    tickets_clean["request_id"].notna()
].copy()

tickets_by_request = (
    ticket_events.groupby("request_id")
    .agg(
        ticket_count=("ticket_id", "nunique"),
        pending_ticket_count=("_is_pending", "sum"),
        escalation_count=("_is_escalated", "sum")
    )
    .reset_index()
)

assert tickets_by_request["request_id"].is_unique

assert (
    tickets_by_request["ticket_count"].sum()
    == tickets_clean["request_id"].notna().sum()
)

if "reassignment_flag" in updates_clean.columns:
    updates_clean["_is_reassignment"] = (
        updates_clean["reassignment_flag"]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin({"yes", "true", "1", "y"})
    )
else:
    updates_clean["_is_reassignment"] = False


if "status_to" in updates_clean.columns:
    updates_clean["_status_to_normalized"] = (
        updates_clean["status_to"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    updates_clean["_is_pending_update"] = (
        updates_clean["_status_to_normalized"].isin(
            {
                "open",
                "pending",
                "reopened",
                "in progress",
                "pending customer",
                "awaiting customer",
                "escalated"
            }
        )
    )
else:
    updates_clean["_is_pending_update"] = False

update_events = updates_clean.loc[
    updates_clean["request_id"].notna()
].copy()

updates_by_request = (
    update_events.groupby("request_id")
    .agg(
        status_update_count=("update_id", "nunique"),
        reassignment_count=("_is_reassignment", "sum"),
        pending_update_count=("_is_pending_update", "sum")
    )
    .reset_index()
)

assert updates_by_request["request_id"].is_unique

assert (
    updates_by_request["status_update_count"].sum()
    == updates_clean["request_id"].notna().sum()
)

def find_orphans(df, event_id, event_type):
    events = df.loc[df["request_id"].notna()].copy()

    orphan_mask = ~events["request_id"].isin(valid_request_ids)

    orphan_rows = events.loc[
        orphan_mask, [event_id, "request_id"]
    ].copy()

    orphan_rows["event_type"] = event_type

    return orphan_rows


orphan_complaints = find_orphans(
    complaints_clean, "complaint_id", "Complaint"
)

orphan_tickets = find_orphans(
    tickets_clean, "ticket_id", "Ticket"
)

orphan_updates = find_orphans(
    updates_clean, "update_id", "Status Update"
)

orphan_events_df = pd.concat(
    [
        orphan_complaints,
        orphan_tickets,
        orphan_updates
    ],
    ignore_index=True
)

missing_link_rows = []

for df, event_id, event_type in [
    (complaints_clean, "complaint_id", "Complaint"),
    (tickets_clean, "ticket_id", "Ticket"),
    (updates_clean, "update_id", "Status Update")
]:
    missing_link_rows.append({
        "event_type": event_type,
        "total_events": len(df),
        "events_with_request_id": int(df["request_id"].notna().sum()),
        "events_with_missing_request_id": int(df["request_id"].isna().sum()),
        "orphan_events": int(
            (
                df["request_id"].notna()
                & ~df["request_id"].isin(valid_request_ids)
            ).sum()
        )
    })

orphan_link_summary = pd.DataFrame(missing_link_rows)

assert (
    orphan_link_summary["total_events"].sum()
    ==
    len(complaints_clean)
    + len(tickets_clean)
    + len(updates_clean)
)

checks = {
    "Complaint IDs unique": complaints_clean["complaint_id"].is_unique,
    "Ticket IDs unique": tickets_clean["ticket_id"].is_unique,
    "Update IDs unique": updates_clean["update_id"].is_unique,

    "Complaint aggregates unique by request": (
        complaints_by_request["request_id"].is_unique
    ),
    "Ticket aggregates unique by request": (
        tickets_by_request["request_id"].is_unique
    ),
    "Update aggregates unique by request": (
        updates_by_request["request_id"].is_unique
    ),

    "Complaint totals reconcile": (
        complaints_by_request["complaint_count"].sum()
        == complaints_clean["request_id"].notna().sum()
    ),
    "Ticket totals reconcile": (
        tickets_by_request["ticket_count"].sum()
        == tickets_clean["request_id"].notna().sum()
    ),
    "Update totals reconcile": (
        updates_by_request["status_update_count"].sum()
        == updates_clean["request_id"].notna().sum()
    )
}

validation_df = pd.DataFrame(
    [
        {"check": name, "result": bool(result)}
        for name, result in checks.items()
    ]
)

print("MILESTONE 07 VALIDATION")

display(validation_df)

print("\nComplaint aggregates:")
display(complaints_by_request.head())

print("\nTicket aggregates:")
display(tickets_by_request.head())

print("\nWorkflow update aggregates:")
display(updates_by_request.head())

print("\nOrphan-link summary:")
display(orphan_link_summary)

print("\nOrphan events preview:")
display(orphan_events_df.head(10))

if validation_df["result"].all():
    print("\nAll Milestone 07 validation checks passed.")
else:
    print("\nSome checks failed. Review the validation table above.")

from pathlib import Path

MILESTONE_07_DIR = Path("hdfc_reports")
MILESTONE_07_DIR.mkdir(parents=True, exist_ok=True)

complaints_by_request.to_csv(
    MILESTONE_07_DIR / "complaints_by_request.csv",
    index=False
)

tickets_by_request.to_csv(
    MILESTONE_07_DIR / "tickets_by_request.csv",
    index=False
)

updates_by_request.to_csv(
    MILESTONE_07_DIR / "updates_by_request.csv",
    index=False
)

orphan_link_summary.to_csv(
    MILESTONE_07_DIR / "orphan_link_summary.csv",
    index=False
)

orphan_events_df.to_csv(
    MILESTONE_07_DIR / "orphan_events_detail.csv",
    index=False
)

print("\nMilestone 07 outputs saved in:", MILESTONE_07_DIR.resolve())

MILESTONE 07: COMPLAINTS, TICKETS & WORKFLOW ANALYSIS
Service requests: 4992
Complaint events: 1500
Support ticket events: 1500
Status update events: 1300
MILESTONE 07 VALIDATION


,check,result
0,Complaint IDs unique,True
1,Ticket IDs unique,True
2,Update IDs unique,True
3,Complaint aggregates unique by request,True
4,Ticket aggregates unique by request,True
5,Update aggregates unique by request,True
6,Complaint totals reconcile,True
7,Ticket totals reconcile,True
8,Update totals reconcile,True



Complaint aggregates:


,request_id,complaint_count,open_complaint_count,negative_sentiment_complaint_count
0,REQ0000014,1,0,0
1,REQ0000017,1,0,1
2,REQ0000025,1,0,1
3,REQ0000026,1,1,1
4,REQ0000028,1,1,1



Ticket aggregates:


,request_id,ticket_count,pending_ticket_count,escalation_count
0,REQ0000012,1,1,0
1,REQ0000016,1,1,0
2,REQ0000020,1,0,1
3,REQ0000025,1,0,1
4,REQ0000030,1,1,0


... (3 outputs left out)

,event_type,total_events,events_with_request_id,events_with_missing_request_id,orphan_events
0,Complaint,1500,1500,0,29
1,Ticket,1500,1500,0,31
2,Status Update,1300,1300,0,30



Orphan events preview:


,complaint_id,request_id,event_type,ticket_id,update_id
0,CMP000029,REQX0000029,Complaint,<NA>,<NA>
1,CMP000044,REQX0000044,Complaint,<NA>,<NA>
2,CMP000056,REQX0000056,Complaint,<NA>,<NA>
3,CMP000202,REQX0000202,Complaint,<NA>,<NA>
4,CMP000203,REQX0000203,Complaint,<NA>,<NA>
5,CMP000229,REQX0000229,Complaint,<NA>,<NA>
6,CMP000232,REQX0000232,Complaint,<NA>,<NA>
7,CMP000340,REQX0000340,Complaint,<NA>,<NA>
8,CMP000490,REQX0000490,Complaint,<NA>,<NA>
9,CMP000501,REQX0000501,Complaint,<NA>,<NA>



All Milestone 07 validation checks passed.

Milestone 07 outputs saved in: /content/hdfc_reports


## Milestone 08 · Build the consolidated request view

**From the team:** Agents should be able to investigate each service request without manually joining six files.

**Your responsibility:** Detect repeated customer/category requests; left-join aggregated event evidence to enriched requests, convert absent counts to zero, and verify row and key reconciliation.

**Handover evidence:** `final_requests_df` with one row per request and `repeat_customer_category` (boolean); all original request IDs retained.

In [10]:
import pandas as pd
import numpy as np

def get_existing_dataframe(names, description):
    """Return the first available DataFrame from a list of names."""
    for name in names:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame):
            print(f"{description}: using '{name}'")
            return obj.copy(), name

    raise NameError(
        f"No DataFrame found for {description}. "
        f"Expected one of these names: {names}"
    )


base_df, base_name = get_existing_dataframe(
    ["requests_enriched", "service_requests_cleaned", "requests_clean"],
    "Request base"
)

if "request_id" not in base_df.columns:
    raise KeyError("The request base DataFrame must contain 'request_id'.")

if base_df["request_id"].isna().any():
    raise ValueError("The cleaned request base contains missing request_id values.")

if base_df["request_id"].duplicated().any():
    raise ValueError(
        "The request base has duplicate request_id values. "
        "Fix duplicates before creating the consolidated view."
    )

original_request_ids = base_df["request_id"].reset_index(drop=True).copy()
original_request_id_set = set(original_request_ids.tolist())

def merge_dimension(base, candidate_names, key, label, desired_columns):
    """Left-join a lookup without multiplying request rows."""
    dimension = None
    dimension_name = None

    for name in candidate_names:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame):
            dimension = obj.copy()
            dimension_name = name
            break

    if dimension is None or key not in dimension.columns:
        print(f"{label} lookup unavailable; retaining existing request data.")
        return base

    columns_to_add = [
        col for col in desired_columns
        if col in dimension.columns and col not in base.columns
    ]

    if not columns_to_add:
        print(f"{label} details already present or no additional columns available.")
        return base

    lookup = dimension[[key] + columns_to_add].copy()

    lookup = lookup.dropna(subset=[key])

    if lookup[key].duplicated().any():
        raise ValueError(
            f"{dimension_name} contains duplicate '{key}' values. "
            "Resolve lookup duplicates before merging."
        )

    before_rows = len(base)

    base = base.merge(
        lookup,
        on=key,
        how="left",
        validate="many_to_one"
    )

    assert len(base) == before_rows, (
        f"{label} lookup unexpectedly changed the number of request rows."
    )

    print(f"{label} lookup merged successfully.")
    return base


final_requests_df = base_df.copy()

final_requests_df = merge_dimension(
    final_requests_df,
    ["customers_clean", "customers"],
    "customer_id",
    "Customer",
    [
        "customer_name",
        "customer_segment",
        "account_type",
        "relationship_age_days",
        "preferred_branch_id"
    ]
)

final_requests_df = merge_dimension(
    final_requests_df,
    ["branches_clean", "branches"],
    "branch_id",
    "Branch",
    [
        "branch_name",
        "branch_city",
        "branch_state",
        "branch_zone",
        "branch_type"
    ]
)

summary_definitions = [
    (
        ["update_summary", "updates_by_request", "status_update_summary"],
        "Status updates"
    ),
    (
        ["complaint_summary", "complaints_by_request"],
        "Complaints"
    ),
    (
        ["ticket_summary", "tickets_by_request"],
        "Support tickets"
    )
]

used_summaries = []

for candidate_names, label in summary_definitions:
    summary = None
    summary_name = None

    for name in candidate_names:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame):
            summary = obj.copy()
            summary_name = name
            break

    if summary is None:
        print(f"Note: no {label} summary found; its count columns will default to zero.")
        continue

    if "request_id" not in summary.columns:
        raise KeyError(
            f"The {summary_name} DataFrame must contain 'request_id'."
        )

    summary = summary.loc[summary["request_id"].notna()].copy()

    if summary["request_id"].duplicated().any():
        raise ValueError(
            f"{summary_name} has duplicate request_id values. "
            "Aggregate it to one row per request before merging."
        )

    matched_summary = summary[
        summary["request_id"].isin(original_request_id_set)
    ].copy()

    used_summaries.append(
        {
            "label": label,
            "name": summary_name,
            "data": summary,
            "matched_data": matched_summary
        }
    )

    columns_to_merge = [
        col for col in summary.columns
        if col == "request_id" or col not in final_requests_df.columns
    ]

    if len(columns_to_merge) > 1:
        final_requests_df = final_requests_df.merge(
            summary[columns_to_merge],
            on="request_id",
            how="left",
            validate="one_to_one"
        )

    print(f"{label}: processed '{summary_name}'.")

expected_count_columns = [
    "status_update_count",
    "reassignment_count",
    "distinct_status_to_count",
    "invalid_update_date_count",

    "complaint_count",
    "unresolved_complaint_count",
    "escalated_complaint_count",
    "reopened_complaint_count",
    "angry_complaint_count",
    "open_complaint_count",
    "negative_sentiment_complaint_count",

    "ticket_count",
    "active_ticket_count",
    "escalated_ticket_count",
    "ticket_sla_breach_count",
    "ticket_delay_warning_count",
    "slow_resolution_ticket_count",
    "slow_resolved_ticket_count",
    "pending_ticket_count",
    "escalation_count",

    "pending_update_count"
]

for col in expected_count_columns:
    if col not in final_requests_df.columns:
        final_requests_df[col] = 0

    final_requests_df[col] = (
        pd.to_numeric(final_requests_df[col], errors="coerce")
        .fillna(0)
        .astype(int)
    )

for col in final_requests_df.columns:
    if col.endswith("_count") and col not in expected_count_columns:
        final_requests_df[col] = (
            pd.to_numeric(final_requests_df[col], errors="coerce")
            .fillna(0)
            .astype(int)
        )

required_repeat_columns = {"customer_id", "request_category"}

if required_repeat_columns.issubset(final_requests_df.columns):
    valid_repeat_key = (
        final_requests_df["customer_id"].notna()
        & final_requests_df["request_category"].notna()
    )

    repeat_counts = (
        final_requests_df.loc[valid_repeat_key]
        .groupby(
            ["customer_id", "request_category"],
            dropna=False
        )["request_id"]
        .transform("size")
    )

    final_requests_df["repeat_customer_category"] = False
    final_requests_df.loc[
        valid_repeat_key, "repeat_customer_category"
    ] = repeat_counts.gt(1).to_numpy()

else:
    missing = required_repeat_columns - set(final_requests_df.columns)
    raise KeyError(
        f"Cannot calculate repeat_customer_category. "
        f"Missing columns: {sorted(missing)}"
    )

final_requests_df["repeat_customer_category"] = (
    final_requests_df["repeat_customer_category"].astype(bool)
)

row_count_matches = len(final_requests_df) == len(base_df)

final_request_id_set = set(final_requests_df["request_id"].tolist())

request_ids_match = (
    final_request_id_set == original_request_id_set
    and final_requests_df["request_id"].is_unique
    and final_requests_df["request_id"].notna().all()
)

request_id_order_matches = (
    final_requests_df["request_id"].reset_index(drop=True)
    .equals(original_request_ids)
)

activity_results = []

for item in used_summaries:
    summary = item["matched_data"]

    check = {
        "activity": item["label"],
        "summary_table": item["name"],
        "matched_summary_rows": len(summary),
        "unique_request_keys": summary["request_id"].nunique(),
        "keys_unique": summary["request_id"].is_unique,
        "keys_belong_to_request_base": summary["request_id"].isin(
            original_request_id_set
        ).all()
    }

    activity_results.append(check)

reconciliation_checks = pd.DataFrame(activity_results)

print("MILESTONE 08 — RECONCILIATION RESULTS")

print(f"Base request rows:          {len(base_df):,}")
print(f"Consolidated request rows:  {len(final_requests_df):,}")
print(f"Unique request IDs:         {final_requests_df['request_id'].nunique():,}")
print(f"Row count matches:          {row_count_matches}")
print(f"Request IDs reconcile:      {request_ids_match}")
print(f"Original ID order retained: {request_id_order_matches}")
print(
    "repeat_customer_category dtype:",
    final_requests_df["repeat_customer_category"].dtype
)

if not reconciliation_checks.empty:
    display(reconciliation_checks)

assert row_count_matches, "FAIL: Consolidation changed the request row count."
assert request_ids_match, "FAIL: Request IDs were lost, added, or duplicated."
assert request_id_order_matches, "FAIL: Original request ID order changed."
assert final_requests_df["repeat_customer_category"].notna().all()
assert final_requests_df["repeat_customer_category"].map(
    lambda value: isinstance(value, (bool, np.bool_))
).all(), "FAIL: repeat_customer_category must contain Boolean values."

if not reconciliation_checks.empty:
    assert reconciliation_checks["keys_unique"].all(), (
        "FAIL: An activity summary has duplicate request keys."
    )
    assert reconciliation_checks["keys_belong_to_request_base"].all(), (
        "FAIL: Unexpected activity-summary request keys."
    )

consolidated_service_requests = final_requests_df.copy()

print("\nSUCCESS: Milestone 08 consolidation checks passed.")
print("Output DataFrame: final_requests_df")
print("Compatibility alias: consolidated_service_requests")

display(final_requests_df.head())

Request base: using 'requests_enriched'
Customer details already present or no additional columns available.
Branch details already present or no additional columns available.
Status updates: processed 'updates_by_request'.
Complaints: processed 'complaints_by_request'.
Support tickets: processed 'tickets_by_request'.
MILESTONE 08 — RECONCILIATION RESULTS
Base request rows:          4,992
Consolidated request rows:  4,992
Unique request IDs:         4,992
Row count matches:          True
Request IDs reconcile:      True
Original ID order retained: True
repeat_customer_category dtype: bool


,activity,summary_table,matched_summary_rows,unique_request_keys,keys_unique,keys_belong_to_request_base
0,Status updates,updates_by_request,1110,1110,True,True
1,Complaints,complaints_by_request,1276,1276,True,True
2,Support tickets,tickets_by_request,1269,1269,True,True



SUCCESS: Milestone 08 consolidation checks passed.
Output DataFrame: final_requests_df
Compatibility alias: consolidated_service_requests


,request_id,customer_id,branch_id,request_created_date,request_created_time,request_category,request_subcategory,channel,request_status,sla_hours,...,escalated_complaint_count,reopened_complaint_count,angry_complaint_count,active_ticket_count,escalated_ticket_count,ticket_sla_breach_count,ticket_delay_warning_count,slow_resolution_ticket_count,slow_resolved_ticket_count,repeat_customer_category
0,REQ0000009,CUST001111,BR0566,2026-04-27,04:20:00,Complaint,Service Delay,Branch,in_progress,72,...,0,0,0,0,0,0,0,0,0,True
1,REQ0000010,CUST000649,BR0980,2026-05-17,22:31:00,Card,Failed ATM Withdrawal,Whatsapp,pending customer,36,...,0,0,0,0,0,0,0,0,0,True
2,REQ0000011,CUST000817,BR0045,2026-06-19,04:47:00,Digital Banking,NetBanking Login,Email,closed,24,...,0,0,0,0,0,0,0,0,0,False
3,REQ0000012,CUST000212,BR0335,2026-04-20,03:36:00,Complaint,Fraud Concern,Email,in_progress,72,...,0,0,0,0,0,0,0,0,0,True
4,REQ0000013,CUST001137,BR0081,2026-06-13,07:18:00,Account,Balance Mismatch,Netbanking,closed,48,...,0,0,0,0,0,0,0,0,0,True


## Milestone 09 · Create transparent service triage

**From the team:** Supervisors need an explainable triage queue without a black-box model.

**Your responsibility:** Create independent signals for SLA breach, open/severe complaint, escalation and repeated customer/category requests; assign High for 2+ signals, Medium for 1, Low otherwise. Generate a practical recommended action.

**Handover evidence:** `final_requests_df` with `priority_score`, `request_priority` (High/Medium/Low) and `recommended_action`; show priority counts.

In [11]:
import pandas as pd
import numpy as np

if "final_requests_df" not in globals():
    raise NameError(
        "final_requests_df not found. Run Milestone 08 first."
    )

if not isinstance(final_requests_df, pd.DataFrame):
    raise TypeError("final_requests_df must be a pandas DataFrame.")

df = final_requests_df.copy()

required_columns = [
    "request_id",
    "request_status",
    "request_category",
    "customer_id",
    "repeat_customer_category"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Milestone 08 is missing required columns: {missing_columns}"
    )

if df["request_id"].isna().any():
    raise ValueError("request_id contains missing values.")

if df["request_id"].duplicated().any():
    raise ValueError(
        "Duplicate request IDs detected. Fix Milestone 08 first."
    )

def numeric_column(data, column):
    """Return a numeric column, using zero when it is absent."""
    if column not in data.columns:
        return pd.Series(0, index=data.index, dtype="int64")

    return pd.to_numeric(
        data[column], errors="coerce"
    ).fillna(0)


def boolean_column(data, column):
    """Convert a flag column to a reliable Boolean Series."""
    if column not in data.columns:
        return pd.Series(False, index=data.index, dtype=bool)

    values = data[column]

    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)

    normalized = values.astype("string").str.strip().str.lower()

    return normalized.isin(
        ["true", "1", "yes", "y", "t"]
    )

if "open_sla_breach" in df.columns:
    sla_signal = boolean_column(df, "open_sla_breach")

elif "open_sla_breach_flag" in df.columns:
    sla_signal = boolean_column(df, "open_sla_breach_flag")

elif "sla_breach_flag" in df.columns:
    sla_signal = boolean_column(df, "sla_breach_flag")

elif "sla_breached" in df.columns:
    sla_signal = boolean_column(df, "sla_breached")

else:
    sla_signal = pd.Series(False, index=df.index, dtype=bool)

df["signal_sla_breach"] = sla_signal.astype(bool)

open_complaint_count = numeric_column(
    df, "unresolved_complaint_count"
)

if (
    "unresolved_complaint_count" not in df.columns
    and "open_complaint_count" in df.columns
):
    open_complaint_count = numeric_column(
        df, "open_complaint_count"
    )

explicit_severe_count = numeric_column(
    df, "severe_open_complaint_count"
)

explicit_severe_flag = boolean_column(
    df, "severe_open_complaint_flag"
)

df["signal_severe_open_complaint"] = (
    open_complaint_count.gt(0)
    | explicit_severe_count.gt(0)
    | explicit_severe_flag
).astype(bool)

escalated_ticket_count = numeric_column(
    df, "escalated_ticket_count"
)

if (
    "escalated_ticket_count" not in df.columns
    and "escalation_count" in df.columns
):
    escalated_ticket_count = numeric_column(
        df, "escalation_count"
    )

escalated_complaint_count = numeric_column(
    df, "escalated_complaint_count"
)

status_escalated = (
    df["request_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("escalated")
    .fillna(False)
)

df["signal_escalated_ticket"] = (
    escalated_ticket_count.gt(0)
    | escalated_complaint_count.gt(0)
    | status_escalated
).astype(bool)

df["signal_repeat_request"] = boolean_column(
    df, "repeat_customer_category"
).astype(bool)

signal_columns = [
    "signal_sla_breach",
    "signal_severe_open_complaint",
    "signal_escalated_ticket",
    "signal_repeat_request"
]

for col in signal_columns:
    df[col] = df[col].fillna(False).astype(bool)

df["priority_score"] = (
    df[signal_columns].astype(int).sum(axis=1)
).astype(int)

df["request_priority"] = np.select(
    [
        df["priority_score"].ge(2),
        df["priority_score"].eq(1)
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)

df["request_priority"] = pd.Categorical(
    df["request_priority"],
    categories=["High", "Medium", "Low"],
    ordered=True
)

def recommend_action(row):
    score = int(row["priority_score"])
    priority = str(row["request_priority"])

    actions = []

    if row["signal_sla_breach"]:
        actions.append(
            "Review SLA breach and arrange resolution immediately"
        )

    if row["signal_severe_open_complaint"]:
        actions.append(
            "Review open complaint and confirm its severity"
        )

    if row["signal_escalated_ticket"]:
        actions.append(
            "Contact the responsible team and review escalation"
        )

    if row["signal_repeat_request"]:
        actions.append(
            "Investigate repeat requests and resolve the underlying issue"
        )

    if priority == "High":
        return (
            "HIGH PRIORITY: Assign an owner and supervise resolution. "
            + "; ".join(actions)
            + "."
        )

    if priority == "Medium":
        return (
            "MEDIUM PRIORITY: Review and follow up promptly. "
            + "; ".join(actions)
            + "."
        )

    return (
        "LOW PRIORITY: Continue routine processing, monitor SLA, "
        "and update the customer as needed."
    )


df["recommended_action"] = df.apply(
    recommend_action, axis=1
)

final_requests_df = df.copy()

consolidated_service_requests = final_requests_df.copy()

priority_order = ["High", "Medium", "Low"]

priority_counts = (
    final_requests_df["request_priority"]
    .value_counts()
    .reindex(priority_order, fill_value=0)
    .rename_axis("request_priority")
    .reset_index(name="total_requests")
)

priority_counts["percentage"] = (
    priority_counts["total_requests"]
    / max(len(final_requests_df), 1)
    * 100
).round(2)

print("MILESTONE 09 — TRANSPARENT SERVICE TRIAGE")

print(f"Total requests: {len(final_requests_df):,}")
print(
    "Total priority signals:",
    len(signal_columns)
)

print("\nPriority counts:")
display(priority_counts)

print("\nSignal summary:")
signal_summary = pd.DataFrame({
    "signal": signal_columns,
    "requests_flagged": [
        int(final_requests_df[col].sum())
        for col in signal_columns
    ]
})
display(signal_summary)

print("\nSample triage queue:")
display(
    final_requests_df[
        [
            "request_id",
            "request_category",
            "priority_score",
            "request_priority",
            "recommended_action"
        ]
    ]
    .sort_values(
        ["priority_score", "request_id"],
        ascending=[False, True]
    )
    .head(15)
)

assert final_requests_df["request_id"].is_unique, (
    "FAIL: Duplicate request IDs detected."
)

assert final_requests_df["priority_score"].between(0, 4).all(), (
    "FAIL: Priority scores must range from 0 to 4."
)

assert final_requests_df["priority_score"].equals(
    final_requests_df[signal_columns].astype(int).sum(axis=1)
), "FAIL: Priority score does not equal the sum of the four signals."

expected_priority = np.select(
    [
        final_requests_df["priority_score"].ge(2),
        final_requests_df["priority_score"].eq(1)
    ],
    ["High", "Medium"],
    default="Low"
)

assert (
    final_requests_df["request_priority"]
    .astype(str)
    .reset_index(drop=True)
    .equals(pd.Series(expected_priority, dtype="object"))
), "FAIL: Priority labels do not match the agreed thresholds."

assert final_requests_df["recommended_action"].notna().all()
assert final_requests_df["recommended_action"].str.strip().ne("").all()

assert priority_counts["total_requests"].sum() == len(
    final_requests_df
), "FAIL: Priority counts do not reconcile to total requests."

print("\nSUCCESS: Milestone 09 validation checks passed.")
print("Output DataFrame: final_requests_df")
print("Required columns: priority_score, request_priority, recommended_action")


MILESTONE 09 — TRANSPARENT SERVICE TRIAGE
Total requests: 4,992
Total priority signals: 4

Priority counts:


,request_priority,total_requests,percentage
0,High,1470,29.45
1,Medium,2328,46.63
2,Low,1194,23.92



Signal summary:


,signal,requests_flagged
0,signal_sla_breach,2629
1,signal_severe_open_complaint,0
2,signal_escalated_ticket,323
3,signal_repeat_request,2484



Sample triage queue:


,request_id,request_category,priority_score,request_priority,recommended_action
47,REQ0000056,Card,3,High,HIGH PRIORITY: Assign an owner and supervise r...
129,REQ0000138,Kyc,3,High,HIGH PRIORITY: Assign an owner and supervise r...
130,REQ0000139,Complaint,3,High,HIGH PRIORITY: Assign an owner and supervise r...
156,REQ0000165,Account,3,High,HIGH PRIORITY: Assign an owner and supervise r...
195,REQ0000204,Loan,3,High,HIGH PRIORITY: Assign an owner and supervise r...
209,REQ0000218,Kyc,3,High,HIGH PRIORITY: Assign an owner and supervise r...
212,REQ0000221,Card,3,High,HIGH PRIORITY: Assign an owner and supervise r...
245,REQ0000254,Complaint,3,High,HIGH PRIORITY: Assign an owner and supervise r...
382,REQ0000391,Kyc,3,High,HIGH PRIORITY: Assign an owner and supervise r...
384,REQ0000393,Card,3,High,HIGH PRIORITY: Assign an owner and supervise r...



SUCCESS: Milestone 09 validation checks passed.
Output DataFrame: final_requests_df
Required columns: priority_score, request_priority, recommended_action


## Milestone 10 · Hand over operational reports

**From the team:** The service lead needs a readable request queue and branch metrics without running Python.

**Your responsibility:** Create ordered detailed/priority/branch reports; export CSV and Excel; write 3–5 evidence-based observations and acknowledge dataset limitations.

**Handover evidence:** `final_report`, `priority_summary`, `branch_sla_summary`; export `hdfc_request_resolution_report.csv`, `hdfc_priority_summary.csv`, `hdfc_branch_sla_summary.xlsx`.

In [12]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

if "final_requests_df" not in globals():
    raise NameError(
        "final_requests_df not found. Run Milestone 08 and all "
        "subsequent milestones first."
    )

final_requests_df = final_requests_df.copy()

if final_requests_df.empty:
    raise ValueError("final_requests_df is empty.")

if "request_id" not in final_requests_df.columns:
    raise ValueError("request_id column is missing.")

if "request_priority" not in final_requests_df.columns:
    raise ValueError(
        "request_priority is missing. Run Milestone 09 first."
    )

if final_requests_df["request_id"].isna().any():
    raise ValueError("Some requests have missing request_id values.")

if not final_requests_df["request_id"].is_unique:
    raise ValueError("Duplicate request_id values found.")

REPORT_DIR = "hdfc_final_reports"
Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)

final_report = final_requests_df.copy()

required_priority_columns = [
    "priority_score",
    "request_priority",
    "recommended_action"
]

missing_priority_columns = [
    col for col in required_priority_columns
    if col not in final_report.columns
]

if missing_priority_columns:
    raise ValueError(
        f"Milestone 09 output columns missing: {missing_priority_columns}"
    )

if "sla_df" not in globals():
    raise NameError("sla_df not found. Run Milestone 05 first.")

sla_required = [
    "request_id",
    "elapsed_hours",
    "sla_hours",
    "sla_breached",
    "open_sla_breach",
    "sla_date_issue"
]

missing_sla = [
    col for col in sla_required
    if col not in sla_df.columns
]

if missing_sla:
    raise ValueError(
        f"Required SLA columns missing from sla_df: {missing_sla}"
    )

sla_fields = [
    "request_id",
    "elapsed_hours",
    "sla_hours",
    "sla_breached",
    "open_sla_breach",
    "sla_date_issue"
]

sla_to_merge = sla_df[
    [
        col for col in sla_fields
        if col == "request_id" or col not in final_report.columns
    ]
].copy()

if not sla_to_merge["request_id"].is_unique:
    raise ValueError("sla_df contains duplicate request_id values.")

final_report = final_report.merge(
    sla_to_merge,
    on="request_id",
    how="left",
    validate="one_to_one"
)

final_report = final_report.sort_values(
    ["priority_score", "request_id"],
    ascending=[False, True],
    kind="stable"
).reset_index(drop=True)

priority_summary = (
    final_requests_df
    .groupby("request_priority", dropna=False)
    .size()
    .reset_index(name="total_requests")
)

priority_summary["percentage_of_requests"] = (
    priority_summary["total_requests"]
    / len(final_requests_df)
    * 100
).round(2)

if "requests_enriched" not in globals():
    raise NameError("requests_enriched not found. Run Milestone 06.")

if "branches_clean" not in globals():
    raise NameError("branches_clean not found. Run Milestone 04.")

branch_source = requests_enriched.copy()

if "branch_id" not in branch_source.columns:
    raise ValueError("branch_id missing from requests_enriched.")

if "branch_id" not in branches_clean.columns:
    raise ValueError("branch_id missing from branches_clean.")

if "branch_sla_summary" not in globals():
    raise NameError("branch_sla_summary not found. Run Milestone 06.")

branch_sla_summary = branch_sla_summary.copy()

if not {"total_requests", "breach_rate"}.issubset(
    branch_sla_summary.columns
):
    raise ValueError(
        "Milestone 06 branch_sla_summary must contain total_requests "
        "and breach_rate. Fix Milestone 06 before exporting."
    )

if branch_sla_summary["total_requests"].sum() != len(requests_clean):
    raise ValueError(
        "Branch request totals do not reconcile with requests_clean. "
        "Fix Milestone 06 before exporting."
    )

if not branch_sla_summary["breach_rate"].dropna().between(0, 1).all():
    raise ValueError(
        "breach_rate must be a proportion between 0 and 1."
    )

report_path = Path(REPORT_DIR) / "hdfc_request_resolution_report.csv"
priority_path = Path(REPORT_DIR) / "hdfc_priority_summary.csv"
branch_path = Path(REPORT_DIR) / "hdfc_branch_sla_summary.xlsx"

final_report.to_csv(report_path, index=False)
priority_summary.to_csv(priority_path, index=False)

with pd.ExcelWriter(branch_path, engine="openpyxl") as writer:
    branch_sla_summary.to_excel(
        writer,
        sheet_name="Branch SLA Summary",
        index=False
    )

    pd.DataFrame({
        "Field": [
            "Project",
            "Milestone",
            "Total service requests",
            "Interpretation"
        ],
        "Value": [
            "HDFC Bank Service Request SLA Analyzer",
            "Milestone 10",
            len(final_report),
            "Educational project dataset; not actual bank operational statistics."
        ]
    }).to_excel(
        writer,
        sheet_name="Report Notes",
        index=False
    )

print("MILESTONE 10: FINAL REPORTS GENERATED")

print("\nFinal report rows:", len(final_report))
print("Unique request IDs:", final_report["request_id"].nunique())

print("\nPriority summary:")
display(priority_summary)

print("\nBranch SLA summary:")
display(branch_sla_summary)

print("\nGenerated files:")

for path in [report_path, priority_path, branch_path]:
    print(f"{path}: {'EXISTS' if path.is_file() else 'MISSING'}")

assert report_path.is_file()
assert priority_path.is_file()
assert branch_path.is_file()

assert len(final_report) == len(final_requests_df)
assert final_report["request_id"].is_unique

print("\nAll three required report files exist.")
print("Milestone 10 completed.")


/tmp/ipykernel_7080/2611952754.py:109: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("request_priority", dropna=False)


MILESTONE 10: FINAL REPORTS GENERATED

Final report rows: 4992
Unique request IDs: 4992

Priority summary:


,request_priority,total_requests,percentage_of_requests
0,High,1470,29.45
1,Medium,2328,46.63
2,Low,1194,23.92



Branch SLA summary:


,branch_id,branch_reporting_name,city_reporting,zone_reporting,total_requests,breached_requests,known_sla_requests,missing_branch_metadata,sla_breached_requests,breach_rate
0,BR0001,HDFC Ahmedabad Service Branch 0001,Ahmedabad,West,6,5,6,False,5,0.833333
1,BR0002,HDFC Kanpur Service Branch 0002,Kanpur,North,3,3,3,False,3,1.000000
2,BR0003,HDFC Salem Service Branch 0003,Salem,South,7,7,7,False,7,1.000000
3,BR0004,HDFC Faridabad Service Branch 0004,Faridabad,North,6,5,6,False,5,0.833333
4,BR0005,HDFC Mysuru Service Branch 0005,Mysuru,South,5,4,5,False,4,0.800000
...,...,...,...,...,...,...,...,...,...,...
1001,BRX0186,Missing Branch Metadata,Unknown City,Unknown Zone,1,1,1,True,1,1.000000
1002,BRX0187,Missing Branch Metadata,Unknown City,Unknown Zone,1,1,1,True,1,1.000000
1003,BRX0188,Missing Branch Metadata,Unknown City,Unknown Zone,1,0,1,True,0,0.000000
1004,BRX0189,Missing Branch Metadata,Unknown City,Unknown Zone,1,1,1,True,1,1.000000



Generated files:
hdfc_final_reports/hdfc_request_resolution_report.csv: EXISTS
hdfc_final_reports/hdfc_priority_summary.csv: EXISTS
hdfc_final_reports/hdfc_branch_sla_summary.xlsx: EXISTS

All three required report files exist.
Milestone 10 completed.


### Handover notes · What should the Support Lead know?
Write **3–5 observations** based on numbers in your actual outputs. Include one operational implication and at least one data-quality limitation.

**Your findings:**

-
-
-

### Final milestone · Automated assessment
**Important:** Colab retains previously created variables and files. After changing or deleting any milestone code, choose **Runtime → Restart session**, then **Run all** before trusting the self-check. Running only the assessment cell can show incorrect passes. For official grading, the instructor must execute your saved `.ipynb` in a fresh kernel with clean input files.

Run the self-check **after all ten milestones**. It creates a scorecard for your notebook using observable DataFrame outputs and files. It does **not** require an answer key to be shared with students.

**How to read results:** `Pass` means the available output meets the named structural or business-rule check; `Review` means something is absent or differs from the expected contract. This is **formative self-checking**, not a secure examination: notebook code can be modified and passing checks do not certify every analysis table or commentary. For official grading, retain a human review of findings, data quality decisions, and a sample of calculations.

**Student instruction:** Run all cells from the top before evaluating. Do not edit the assessment cell.

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

_scope = globals()
_checks = []

def check(task, criterion, fn):
    try:
        passed = bool(fn())
        msg = 'Meets check' if passed else 'Missing or inconsistent output'
    except Exception as ex:
        passed = False
        msg = f'Cannot verify: {type(ex).__name__}'
    _checks.append({'Milestone': task, 'Check': criterion, 'Result': 'Pass' if passed else 'Review', 'Points': 5 if passed else 0, 'Feedback': msg})

def df(name):
    obj = _scope.get(name)
    if not isinstance(obj, pd.DataFrame):
        raise ValueError(f'{name} is not a DataFrame')
    return obj

def has_cols(name, cols):
    return set(cols).issubset(df(name).columns)

def unique_nonnull(frame, key):
    return key in frame and frame[key].notna().all() and frame[key].is_unique

def numeric(frame, col):
    return col in frame and pd.api.types.is_numeric_dtype(frame[col])

def real_dates(frame, cols):
    return all(col in frame and pd.api.types.is_datetime64_any_dtype(frame[col]) for col in cols)

def expected_counts(name, count_col, source, key):
    a = df(name)
    s = df(source)
    return (has_cols(name,[key,count_col]) and unique_nonnull(a,key)
            and numeric(a,count_col)
            and a[count_col].sum()==s[key].notna().sum())

def event_counts_match():
    m = df('final_requests_df')
    if not unique_nonnull(m,'request_id'): return False
    for source,col in [('complaints_clean','complaint_count'),
                       ('tickets_clean','ticket_count'),
                       ('updates_clean','status_update_count')]:
        s = df(source)
        if col not in m or 'request_id' not in s: return False
        counts = s[s.request_id.isin(m.request_id)].groupby('request_id').size()
        actual = m.set_index('request_id')[col].fillna(0)
        aligned = counts.reindex(actual.index,fill_value=0)
        if not actual.eq(aligned).all(): return False
    return True

def sla_matches():
    s = df('sla_df')
    required = ['request_id','elapsed_hours','sla_hours',
                'sla_breached','open_sla_breach','sla_date_issue']
    if not has_cols('sla_df',required): return False
    if not numeric(s,'elapsed_hours') or not numeric(s,'sla_hours'): return False
    if any(not pd.api.types.is_bool_dtype(s[x]) for x in
           ['sla_breached','open_sla_breach']): return False
    expected = (s.elapsed_hours.gt(s.sla_hours)
                & s.elapsed_hours.notna() & s.sla_hours.notna())
    return (unique_nonnull(s,'request_id')
            and s.sla_breached.eq(expected).all()
            and (~s.open_sla_breach | s.sla_breached).all())

def signals_match():
    m = df('final_requests_df')
    flags = ['signal_sla_breach','signal_severe_open_complaint',
             'signal_escalated_ticket','signal_repeat_request']
    if not has_cols('final_requests_df',flags+['priority_score']): return False
    if any(not pd.api.types.is_bool_dtype(m[x]) for x in flags): return False
    return (numeric(m,'priority_score')
            and m.priority_score.eq(m[flags].sum(axis=1)).all())

def priorities_match():
    m = df('final_requests_df')
    if not has_cols('final_requests_df',
                    ['priority_score','request_priority','recommended_action']):
        return False
    expected = m.priority_score.map(
        lambda x: 'High' if x>=2 else ('Medium' if x==1 else 'Low'))
    return (m.request_priority.eq(expected).all()
            and m.recommended_action.notna().all()
            and m.recommended_action.astype(str).str.strip().ne('').all())

def report_matches():
    r = df('final_report')
    m = df('final_requests_df')
    if not has_cols('final_report',['request_id','request_priority']): return False
    return (unique_nonnull(r,'request_id')
            and len(r)==len(m)
            and set(r.request_id)==set(m.request_id)
            and r.set_index('request_id').request_priority.eq(
                m.set_index('request_id').loc[r.request_id,'request_priority'].to_numpy()).all())

def summary_matches():
    s = df('priority_summary')
    if not has_cols('priority_summary',['request_priority','total_requests']): return False
    expected = df('final_requests_df').request_priority.value_counts()
    got = s.groupby('request_priority').total_requests.sum()
    return (got.reindex(expected.index,fill_value=0).eq(expected).all()
            and got.sum()==len(df('final_requests_df')))

check(1,'Service requests loaded, key fields present',lambda: has_cols('requests_df',['request_id','customer_id','branch_id']))
check(1,'Service request source is non-empty',lambda: len(df('requests_df'))>0 and df('requests_df').request_id.notna().all())
check(2,'Five supporting DataFrames loaded',lambda: all(isinstance(df(n),pd.DataFrame) for n in ['customers_df','branches_df','tickets_df','updates_df','complaints_df']))
check(2,'Supporting event identifiers present',lambda: all(has_cols(n,[k,'request_id']) for n,k in [('tickets_df','ticket_id'),('updates_df','update_id'),('complaints_df','complaint_id')]))
check(3,'One clean record per service request',lambda: unique_nonnull(df('requests_clean'),'request_id') and len(df('requests_clean'))==df('requests_df').request_id.nunique())
check(3,'Request dates, SLA and anomaly flags prepared',lambda: real_dates(df('requests_clean'),['created_at']) and numeric(df('requests_clean'),'sla_hours') and has_cols('requests_clean',['request_date_issue']))
check(4,'Customer and branch dimension keys are unique',lambda: unique_nonnull(df('customers_clean'),'customer_id') and unique_nonnull(df('branches_clean'),'branch_id'))
check(4,'Complaint, ticket and update IDs are unique',lambda: all(unique_nonnull(df(n),k) for n,k in [('complaints_clean','complaint_id'),('tickets_clean','ticket_id'),('updates_clean','update_id')]))
check(5,'SLA calculations follow business rules',sla_matches)
check(5,'Category SLA summary reconciles to requests',lambda: has_cols('category_sla_summary',['total_requests']) and df('category_sla_summary').total_requests.sum()==len(df('sla_df')))
check(6,'Enriched requests preserve request population',lambda: unique_nonnull(df('requests_enriched'),'request_id') and set(df('requests_enriched').request_id)==set(df('requests_clean').request_id))
check(6,'Branch SLA summary reconciles',lambda: has_cols('branch_sla_summary',['total_requests','breach_rate']) and df('branch_sla_summary').total_requests.sum()==len(df('requests_clean')) and df('branch_sla_summary').breach_rate.between(0,1).all())
check(7,'Event aggregates preserve unique request keys',lambda: all(unique_nonnull(df(n),'request_id') for n in ['complaints_by_request','tickets_by_request','updates_by_request']))
check(7,'Aggregated event totals match cleaned sources',lambda: all([expected_counts('complaints_by_request','complaint_count','complaints_clean','request_id'),expected_counts('tickets_by_request','ticket_count','tickets_clean','request_id'),expected_counts('updates_by_request','status_update_count','updates_clean','request_id')]))
check(8,'Master table contains each request exactly once',lambda: unique_nonnull(df('final_requests_df'),'request_id') and set(df('final_requests_df').request_id)==set(df('requests_clean').request_id))
check(8,'Master event counts reconcile',lambda: event_counts_match() and has_cols('final_requests_df',['repeat_customer_category']))
check(9,'Priority signals and scores are consistent',signals_match)
check(9,'Priority classification and actions are consistent',priorities_match)
check(10,'Detailed report and export exist',lambda: report_matches() and (Path(REPORT_DIR)/'hdfc_request_resolution_report.csv').is_file())
check(10,'Priority and branch summaries exported',lambda: summary_matches() and all((Path(REPORT_DIR)/n).is_file() for n in ['hdfc_priority_summary.csv','hdfc_branch_sla_summary.xlsx']))

assessment_df = pd.DataFrame(_checks)
score_by_task = assessment_df.groupby('Milestone',as_index=False).agg(Points=('Points','sum'),Passed=('Result',lambda x:(x=='Pass').sum()))
score_by_task['Maximum'] = 10
score_by_task['Status'] = score_by_task['Passed'].map({0:'Review',1:'Partially verified',2:'Checks passed'})
print('AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE')
display(score_by_task[['Milestone','Points','Maximum','Status']])
print(f'Automated checks score: {assessment_df.Points.sum()}/100')
print('Checks needing attention:')
display(assessment_df.loc[assessment_df.Result=='Review',['Milestone','Check','Feedback']].reset_index(drop=True))

import json as _grade_json
print('HDFC_ASSESSMENT_JSON=' + _grade_json.dumps({'points': int(assessment_df.Points.sum()), 'milestones': score_by_task[['Milestone','Points','Maximum','Status']].to_dict('records'), 'checks': assessment_df[['Milestone','Check','Result','Points']].to_dict('records')}))


AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE


/tmp/ipykernel_7080/4241479346.py:105: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  got = s.groupby('request_priority').total_requests.sum()


,Milestone,Points,Maximum,Status
0,1,10,10,Checks passed
1,2,10,10,Checks passed
2,3,10,10,Checks passed
3,4,10,10,Checks passed
4,5,10,10,Checks passed
5,6,10,10,Checks passed
6,7,10,10,Checks passed
7,8,10,10,Checks passed
8,9,10,10,Checks passed
9,10,10,10,Checks passed


Automated checks score: 100/100
Checks needing attention:


,Milestone,Check,Feedback


HDFC_ASSESSMENT_JSON={"points": 100, "milestones": [{"Milestone": 1, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 2, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 3, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 4, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 5, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 6, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 7, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 8, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 9, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 10, "Points": 10, "Maximum": 10, "Status": "Checks passed"}], "checks": [{"Milestone": 1, "Check": "Service requests loaded, key fields present", "Result": "Pass", "Points": 5}, {"Milestone": 1, "Check": "Service request source is non-empty", "Result": "Pass", "Points

### Submission handover

1. Submit your completed student notebook (`.ipynb`) with the three exported reports and visible assessment output (or share a Colab copy if required).
2. Include 3–5 supported findings and a short explanation of AI usage, assumptions, and unresolved data-quality issues.
3. The ZIP includes six structured business datasets plus `status_notes.txt`; never substitute datasets from another project.
4. Student and instructor notebooks should be run from a fresh session with the supplied original data files.